# ARRGO: Theoretical Foundations

This notebook establishes the mathematical assumptions, properties, and
theoretical conditions required to analyze the correctness, convergence,
and optimality guarantees of ARRGO.

## Theoretical Assumptions

To analyze the correctness, convergence, and optimality properties of ARRGO,
we explicitly state the assumptions under which the theoretical results hold.

The assumptions are divided into two categories:

1. **Baseline assumptions**, which define the optimization problem and the
   validity of the refinement process.
2. **Certified-mode assumptions**, which are required when ARRGO uses rigorous
   uncertainty bounds and optimality certificates.

These assumptions describe the mathematical setting of the analysis. They do
not necessarily represent information that must be explicitly provided to the
algorithm during every run.

### Baseline Assumptions

#### 1. Bounded Search Domain

The optimization domain is a non-empty compact interval

$$
\Omega=[a,b],
\qquad a<b.
$$

Therefore, the search space is bounded and has finite diameter

$$
\operatorname{diam}(\Omega)=b-a.
$$

#### 2. Objective Function Evaluability

The objective function is deterministic and can be evaluated at any valid
point in the search domain:

$$
x\in\Omega
\quad\Longrightarrow\quad
f(x)\in\mathbb{R}.
$$

ARRGO does not require the analytical expression of $f$ to be available.

#### 3. Continuity of the Objective Function

For the theoretical analysis of global optimum existence, the objective
function is assumed to be continuous on $\Omega$:

$$
f\in C(\Omega).
$$

Since $\Omega$ is compact and $f$ is continuous, a global maximum exists:

$$
f^*=\max_{x\in\Omega} f(x).
$$

Thus, the global optimization problem is well-defined.

#### 4. Valid Refinement

Every refinement operation must preserve the validity of the search domain.

For a split of a region

$$
R=[l,r]
$$

at a point $s\in(l,r)$, the resulting regions are

$$
R_L=[l,s],
\qquad
R_R=[s,r].
$$

They must satisfy

$$
R_L\cup R_R=R.
$$

Furthermore, every accepted split satisfies the contraction condition

$$
\max(s-l,r-s)\leq\rho(r-l),
\qquad 0<\rho<1.
$$

#### 5. Valid Function Evaluations

Every function evaluation performed by ARRGO corresponds to a point inside the
search domain:

$$
x_{\mathrm{new}}\in\Omega.
$$

Duplicate evaluations are avoided up to the numerical spatial tolerance
defined by the implementation.

#### 6. Global Selection Fairness

The global region-selection mechanism must not permanently ignore a region that
remains relevant to the optimization process.

In particular, if a region remains capable of containing information that can
affect the global optimization decision, the selection mechanism must allow
that region to be selected for further refinement.

This condition will be formalized later as part of the convergence analysis.

### Certified-Mode Assumptions

The following assumptions are required only when ARRGO operates in certified
mode.

#### 7. Known Valid Lipschitz Bound

The objective function is assumed to satisfy a globally valid Lipschitz
condition with constant $L>0$:

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

The constant $L$ must be a valid upper bound on the true Lipschitz constant of
the objective function.

Under this assumption, ARRGO can construct rigorous lower and upper
function-value bounds from observed function evaluations.

#### 8. Deterministic Exact-Evaluation Model

For the baseline theoretical results, function evaluations are treated as
exact:

$$
y_i=f(x_i).
$$

Measurement noise and stochastic objective functions are outside the scope of
the current theoretical analysis.

### Separation of Assumptions and Guarantees

Not every assumption is required for every ARRGO capability.

In particular:

- Continuity is used to establish the existence of a global optimum on the
  compact domain.
- Contraction is used to establish spatial refinement.
- Global selection fairness is required for the convergence analysis.
- A valid Lipschitz bound is required for rigorous uncertainty bounds and
  certified optimality guarantees.
- Exact evaluations are assumed for the idealized deterministic theoretical
  model.

Therefore, ARRGO distinguishes between **algorithmic behavior** and
**theoretical guarantees**.

$$
\boxed{
\text{Guarantee}
\;\Longrightarrow\;
\text{Assumptions}
+
\text{Valid Refinement}
+
\text{Valid Analysis}
}
$$

## Existence of a Global Optimum

The existence of a global optimum is a fundamental requirement for the
theoretical analysis of ARRGO.

Under the baseline assumptions, the search domain $\Omega$ is compact and the
objective function $f$ is continuous on $\Omega$:

$$
\Omega=[a,b],
\qquad
f\in C(\Omega).
$$

By the Extreme Value Theorem (Weierstrass theorem), a continuous function on a
compact domain attains both its maximum and minimum.

Therefore, there exists at least one point

$$
x^*\in\Omega
$$

such that

$$
f(x^*)=\max_{x\in\Omega}f(x).
$$

We define the global optimal value as

$$
f^*=f(x^*)
=
\max_{x\in\Omega}f(x).
$$

The set of global maximizers is

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

In general, $X^*$ may contain more than one point. ARRGO therefore does not
assume that the global maximizer is unique.

### Why This Matters for ARRGO

The existence result gives ARRGO a well-defined target:

$$
\boxed{
\text{Global optimization target}
=
\max_{x\in\Omega}f(x)
}
$$

This distinction is important because the algorithm may first obtain a
**best-found solution**

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D}f(x_i),
$$

while the true global optimum is

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

Since the evaluated set $\mathcal D$ is generally only a finite subset of
$\Omega$,

$$
f_{\mathrm{best}}\leq f^*.
$$

Equality holds only when the evaluated information is sufficient to identify
a globally optimal value.

Thus, the theoretical analysis must distinguish between:

$$
\boxed{
\text{Best Found Solution}
\neq
\text{True Global Optimum}
}
$$

unless additional conditions provide a valid optimality certificate.

## Continuity and Compactness

Continuity and compactness provide the mathematical foundation for the
existence and stability of the global optimization problem considered by
ARRGO.

### Continuity

The objective function is assumed to be continuous on the search domain:

$$
f\in C(\Omega).
$$

Continuity means that sufficiently small changes in the input produce
sufficiently small changes in the objective value.

Formally, for every $\varepsilon>0$ and every $x\in\Omega$, there exists
$\delta>0$ such that

$$
|x-y|<\delta
\quad\Longrightarrow\quad
|f(x)-f(y)|<\varepsilon.
$$

This property is important for adaptive region refinement because ARRGO
progressively reduces the size of regions.

If a sequence of points satisfies

$$
x_k\to x,
$$

then continuity guarantees

$$
f(x_k)\to f(x).
$$

Therefore, as the spatial resolution of a region increases, the function
values observed within that region become increasingly related to its local
behavior.

### Compactness

The search domain is assumed to be the closed and bounded interval

$$
\Omega=[a,b],
\qquad a<b.
$$

In $\mathbb{R}$, every closed and bounded interval is compact.

Compactness provides two important properties for ARRGO.

First, together with continuity, it guarantees the existence of a global
maximum:

$$
\exists x^*\in\Omega:
\qquad
f(x^*)=\max_{x\in\Omega}f(x).
$$

Second, compactness ensures that the entire optimization problem is contained
inside a finite spatial domain:

$$
\operatorname{diam}(\Omega)=b-a<\infty.
$$

This finite diameter allows ARRGO to reason about spatial refinement through
the sizes of regions.

### Interaction with Region Refinement

Consider a sequence of nested regions

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots
$$

generated by ARRGO.

If the refinement mechanism satisfies the contraction condition

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\,\operatorname{diam}(R_k),
\qquad
0<\rho<1,
$$

then

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\lim_{k\to\infty}\operatorname{diam}(R_k)=0.
$$

Continuity then implies that function values within sufficiently refined
regions become increasingly close whenever their spatial distance becomes
sufficiently small.

This provides the basic connection between ARRGO's spatial refinement and
the mathematical behavior of the objective function:

$$
\boxed{
\text{Compact Domain}
\;+\;
\text{Continuity}
\;+\;
\text{Spatial Contraction}
}
$$

creates the foundation for analyzing increasingly localized regions around
potential optima.

However, continuity and contraction alone do not provide a numerical bound on
the unknown objective values inside a region. Rigorous numerical bounds
require additional assumptions, such as the Lipschitz condition introduced
later.

## Lipschitz Regularity

Continuity guarantees that small changes in the input produce small changes
in the objective value, but it does not provide a quantitative bound on the
magnitude of those changes.

For certified uncertainty estimation, ARRGO may additionally assume that the
objective function satisfies a global Lipschitz condition.

### Lipschitz Condition

A function $f:\Omega\rightarrow\mathbb{R}$ is Lipschitz continuous on $\Omega$
if there exists a constant $L\geq0$ such that

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

The constant $L$ is called a Lipschitz constant of $f$.

A valid Lipschitz constant does not need to be the smallest possible constant.
Any value satisfying the inequality for every pair of points in $\Omega$ is
sufficient for the theoretical guarantees.

### Interpretation

The Lipschitz condition places a quantitative limit on how rapidly the
objective function can change with respect to the input.

For two points $x$ and $y$,

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

Therefore, if the distance between two points is small, the maximum possible
difference between their function values is also bounded.

For example, if an evaluated point $x_i$ has value

$$
y_i=f(x_i),
$$

then for any other point $x\in\Omega$,

$$
|f(x)-y_i|
\leq
L|x-x_i|.
$$

Equivalently,

$$
y_i-L|x-x_i|
\leq
f(x)
\leq
y_i+L|x-x_i|.
$$

These inequalities provide lower and upper bounds on the unknown value
$f(x)$.

### Relation to Region Refinement

Consider a region

$$
R=[l,r].
$$

For any two points $x,y\in R$,

$$
|x-y|
\leq
\operatorname{diam}(R)
=
r-l.
$$

Therefore,

$$
|f(x)-f(y)|
\leq
L(r-l).
$$

As ARRGO refines a region and its diameter decreases, the maximum possible
variation implied by the Lipschitz condition also decreases.

If

$$
\operatorname{diam}(R_k)\to0,
$$

then

$$
L\operatorname{diam}(R_k)\to0.
$$

Thus, spatial contraction directly reduces the maximum function-value
variation allowed by the Lipschitz model.

### Role in Certified ARRGO

The Lipschitz assumption enables ARRGO to transform observed function
evaluations into rigorous bounds over unobserved points.

This allows the algorithm to distinguish between two different concepts:

$$
\boxed{
\text{Observed Information}
}
$$

and

$$
\boxed{
\text{Certified Information About Unobserved Points}
}
$$

The former is available from function evaluations alone, while the latter
requires a valid mathematical assumption such as the Lipschitz condition.

Therefore, Lipschitz continuity is **not required for the basic operation of
ARRGO**. It is an additional assumption used by the certified version of the
framework.

### Important Distinction

The Lipschitz constant used by ARRGO must be a valid upper bound.

An estimated value $\widehat L$ is not automatically a valid Lipschitz
constant:

$$
\widehat L
\not\Rightarrow
\text{Certified Bound}.
$$

If

$$
\widehat L < L_{\mathrm{true}},
$$

the resulting envelopes may fail to contain the true function and therefore
cannot provide a rigorous optimality certificate.

Consequently,

$$
\boxed{
\text{Certified ARRGO}
\Longrightarrow
\text{Valid Lipschitz Bound}
}
$$

while

$$
\boxed{
\text{Basic ARRGO}
\not\Longrightarrow
\text{Lipschitz Assumption}
}
$$

## Validity of Lipschitz-Based Bounds

Assume that $f$ satisfies the Lipschitz condition

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

Suppose ARRGO has evaluated the objective function at the points

$$
\mathcal D_R
=
\{(x_i,y_i)\}_{i=1}^{n},
\qquad
y_i=f(x_i).
$$

For any point $x\in R$, applying the Lipschitz condition to $x$ and each
observed point $x_i$ gives

$$
|f(x)-y_i|
\leq
L|x-x_i|.
$$

Therefore,

$$
-L|x-x_i|
\leq
f(x)-y_i
\leq
L|x-x_i|,
$$

which is equivalent to

$$
y_i-L|x-x_i|
\leq
f(x)
\leq
y_i+L|x-x_i|.
$$

Thus, every observed point provides an independent lower and upper bound on
the unknown value $f(x)$.

### Lower Envelope

Since the lower bound must hold for every observed point,

$$
f(x)
\geq
y_i-L|x-x_i|,
\qquad
i=1,\ldots,n.
$$

Therefore, the strongest lower bound obtained from all observations is

$$
\boxed{
L_R(x)
=
\max_{1\leq i\leq n}
\left[
y_i-L|x-x_i|
\right]
}
$$

and consequently,

$$
L_R(x)\leq f(x).
$$

### Upper Envelope

Similarly, every observed point provides an upper bound

$$
f(x)
\leq
y_i+L|x-x_i|.
$$

The strongest upper bound consistent with all observations is therefore

$$
\boxed{
U_R(x)
=
\min_{1\leq i\leq n}
\left[
y_i+L|x-x_i|
\right]
}
$$

and consequently,

$$
f(x)\leq U_R(x).
$$

### Validity Theorem

Combining the two inequalities gives

$$
\boxed{
L_R(x)
\leq
f(x)
\leq
U_R(x),
\qquad
\forall x\in R.
}
$$

Therefore, the interval

$$
[L_R(x),U_R(x)]
$$

is a valid pointwise enclosure of the objective value at $x$.

### Proof

For every observation $(x_i,y_i)$,

$$
y_i-L|x-x_i|
\leq
f(x).
$$

Since this inequality holds for all $i$, it also holds for their maximum:

$$
\max_i
\left[
y_i-L|x-x_i|
\right]
\leq
f(x).
$$

Hence,

$$
L_R(x)\leq f(x).
$$

Likewise, for every observation,

$$
f(x)
\leq
y_i+L|x-x_i|.
$$

Since this inequality holds for all $i$, it also holds for their minimum:

$$
f(x)
\leq
\min_i
\left[
y_i+L|x-x_i|
\right].
$$

Hence,

$$
f(x)\leq U_R(x).
$$

Combining both results,

$$
L_R(x)
\leq
f(x)
\leq
U_R(x).
$$

Therefore, the Lipschitz-based envelopes are valid bounds on the objective
function over the region.

### Consequence for ARRGO

The validity of these envelopes means that ARRGO can reason about points that
have never been evaluated directly.

The uncertainty at a point can be defined as

$$
u_R(x)
=
U_R(x)-L_R(x).
$$

Because the true function value is enclosed by the two bounds,

$$
f(x)\in[L_R(x),U_R(x)].
$$

Thus, the uncertainty measure is not merely an empirical estimate when the
Lipschitz assumption is valid. It represents a mathematically justified
interval of possible function values.

This establishes the theoretical foundation for ARRGO's certified uncertainty
and optimization-potential mechanisms.

## Spatial Contraction

Spatial contraction is the structural mechanism that guarantees that repeated
region refinement produces increasingly smaller search regions.

Consider a region

$$
R=[l,r]
$$

and a split point

$$
s\in(l,r).
$$

The split produces two child regions

$$
R_L=[l,s],
\qquad
R_R=[s,r].
$$

The contraction condition requires

$$
\max(s-l,r-s)
\leq
\rho(r-l),
\qquad
0<\rho<1.
$$

This condition guarantees that the diameter of every child region is strictly
smaller than the diameter of its parent.

### One-Step Contraction

The diameter of the parent region is

$$
\operatorname{diam}(R)=r-l.
$$

The diameters of the two child regions are

$$
\operatorname{diam}(R_L)=s-l
$$

and

$$
\operatorname{diam}(R_R)=r-s.
$$

By the contraction condition,

$$
\max
\left\{
\operatorname{diam}(R_L),
\operatorname{diam}(R_R)
\right\}
\leq
\rho\operatorname{diam}(R).
$$

Since

$$
0<\rho<1,
$$

we have

$$
\rho\operatorname{diam}(R)
<
\operatorname{diam}(R).
$$

Therefore,

$$
\boxed{
\operatorname{diam}(R_L)<\operatorname{diam}(R)
}
$$

and

$$
\boxed{
\operatorname{diam}(R_R)<\operatorname{diam}(R).
}
$$

Thus, every accepted split produces strictly smaller child regions.

### Repeated Contraction

Consider a sequence of regions generated by repeated refinement:

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots
$$

where each refinement satisfies the contraction condition.

Then

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\operatorname{diam}(R_k).
$$

Applying this relation repeatedly gives

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Because

$$
0<\rho<1,
$$

we have

$$
\lim_{k\to\infty}\rho^k=0.
$$

Consequently,

$$
\boxed{
\lim_{k\to\infty}
\operatorname{diam}(R_k)
=
0.
}
$$

Therefore, any infinite sequence of refinements along a nested path produces
regions whose spatial diameter converges to zero.

### Relation to Function Resolution

Under the Lipschitz assumption,

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

For any two points $x,y\in R_k$,

$$
|x-y|
\leq
\operatorname{diam}(R_k).
$$

Therefore,

$$
|f(x)-f(y)|
\leq
L\operatorname{diam}(R_k).
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

it follows that

$$
L\operatorname{diam}(R_k)\to0.
$$

Hence, the maximum variation allowed by the Lipschitz condition inside the
region also converges to zero.

This establishes the following relationship:

$$
\boxed{
\text{Spatial Contraction}
\Longrightarrow
\text{Decreasing Spatial Uncertainty}
}
$$

and, under a valid Lipschitz bound,

$$
\boxed{
\text{Spatial Contraction}
\Longrightarrow
\text{Decreasing Function-Value Variation}
}
$$

### Role in ARRGO Convergence

Spatial contraction alone does not prove that ARRGO finds the global optimum.

It provides the structural component required for convergence analysis:

$$
\text{Repeated Refinement}
\Longrightarrow
\text{Shrinking Regions}.
$$

To obtain a global optimization guarantee, this property must be combined with
a valid information mechanism and a global region-selection condition.

Therefore, the convergence argument will later rely on the combination

$$
\boxed{
\text{Spatial Contraction}
+
\text{Information Refinement}
+
\text{Global Selection}
}
$$

rather than on contraction alone.

## Information Resolution

ARRGO does not refine regions only to reduce their spatial size. The purpose of
spatial refinement is to obtain increasingly informative observations about the
objective function.

Let a region be represented by

$$
R=(I,\mathcal D_R),
$$

where $I$ is its spatial domain and $\mathcal D_R$ is the set of available
function evaluations inside or relevant to the region.

The information state of a region can be represented abstractly as

$$
S_R=(\mathcal D_R,B_R,\mathcal Q_R),
$$

where:

- $\mathcal D_R$ represents observed function values,
- $B_R$ represents the observed behavioral profile,
- $\mathcal Q_R$ represents unresolved information.

### Spatial Resolution

The spatial resolution of a region is related to its diameter:

$$
\operatorname{diam}(R).
$$

Under the contraction condition,

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0),
\qquad
0<\rho<1.
$$

Therefore,

$$
\operatorname{diam}(R_k)\to0.
$$

A smaller region provides a more localized spatial context for interpreting
function evaluations.

### Information Resolution

Let

$$
\mathcal Q_R
=
[
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
]
$$

denote the unresolved information profile of a region.

Refinement is considered successful when it reduces the information relevant
to the current optimization decision.

For a refinement action $A$, let the resulting information state be

$$
S_R^A.
$$

The corresponding unresolved information profile is

$$
\mathcal Q_R^A.
$$

The conceptual information improvement produced by the action is therefore

$$
\Delta\mathcal Q(A\mid R)
=
\mathcal Q_R-\mathcal Q_R^A.
$$

This expression represents a reduction in unresolved information rather than a
single numerical objective.

### Sampling and Information Resolution

A sampling operation adds a new observation:

$$
\mathcal D_R'
=
\mathcal D_R
\cup
\{(x_{\mathrm{new}},f(x_{\mathrm{new}}))\}.
$$

The new observation may reduce uncertainty about:

- local function behavior,
- directional changes,
- candidate extrema,
- spatial coverage,
- optimization potential.

Therefore,

$$
\boxed{
\text{Sampling}
\Longrightarrow
\text{Information Refinement}
}
$$

### Splitting and Information Resolution

A splitting operation changes the spatial structure of the problem:

$$
R\rightarrow\{R_L,R_R\}.
$$

The observations associated with the parent region can then be interpreted
within smaller spatial contexts.

This may reveal differences between subregions that were not sufficiently
visible at the parent scale.

Therefore,

$$
\boxed{
\text{Splitting}
\Longrightarrow
\text{Structural Information Refinement}
}
$$

### Information Resolution and Lipschitz Uncertainty

In certified mode, the Lipschitz condition provides

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

For points inside a region $R$,

$$
|f(x)-f(y)|
\leq
L\operatorname{diam}(R).
$$

Thus, as the region contracts,

$$
\operatorname{diam}(R)\to0
$$

and consequently,

$$
L\operatorname{diam}(R)\to0.
$$

This means that the maximum function-value variation permitted solely by the
regional diameter also decreases.

Therefore, spatial refinement contributes directly to information resolution
under the Lipschitz assumption.

### Important Limitation

Spatial contraction does not automatically imply that every aspect of the
objective function has been sufficiently observed.

A region can be spatially small while still containing unresolved information
if, for example, its observations are insufficient to characterize the local
behavior relevant to the optimization decision.

Therefore,

$$
\boxed{
\text{Small Region}
\neq
\text{Automatically Sufficient Information}
}
$$

ARRGO must evaluate both spatial resolution and information resolution.

The theoretical objective is therefore not simply

$$
\operatorname{diam}(R)\to0,
$$

but rather

$$
\boxed{
\text{Spatial Resolution}
+
\text{Information Resolution}
}
$$

as the foundation for subsequent convergence and optimality analysis.

## Optimization Potential Bound

The purpose of an optimization-potential bound is to quantify how good a
region could still be under the information currently available to ARRGO.

Consider a region

$$
R=[l,r]
$$

with a valid upper envelope

$$
f(x)\le U_R(x),
\qquad
\forall x\in R.
$$

Since the inequality holds for every point in the region, it also holds at a
global maximizer contained in that region.

### Regional Upper Potential

Define the optimization potential of the region as

$$
\boxed{
P(R)
=
\max_{x\in R} U_R(x)
}
$$

Because

$$
f(x)\le U_R(x),
\qquad
\forall x\in R,
$$

we have

$$
\max_{x\in R}f(x)
\leq
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\boxed{
\max_{x\in R}f(x)\leq P(R)
}
$$

and $P(R)$ is a valid upper bound on the best objective value that can occur
inside region $R$.

### Relation to the Global Optimum

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

be the global optimal value.

If the global optimizer belongs to a region $R^*$, then

$$
f^*
=
\max_{x\in R^*}f(x).
$$

By the regional upper bound,

$$
f^*
\leq
P(R^*).
$$

Therefore, the potential of the region containing a global maximizer is always
at least as large as the true global optimum:

$$
\boxed{
f^*\leq P(R^*)
}
$$

This property makes regional potential useful for global optimization.

### Global Potential Bound

For a collection of regions

$$
\mathcal R=\{R_1,\ldots,R_m\},
$$

define the global potential as

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal R}P(R).
$$

Because the search domain is covered by the regions,

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R}R,
$$

the global optimum is contained in at least one represented region.

Consequently,

$$
\boxed{
f^*
\leq
P_{\mathrm{global}}
}
$$

and the global potential provides an upper bound on the unknown optimum.

### Relation to the Incumbent

ARRGO maintains the best function value obtained from actual evaluations:

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D}f(x_i).
$$

Since every evaluated point belongs to the search domain,

$$
f_{\mathrm{best}}
\leq
f^*.
$$

Combining this with the global potential bound gives

$$
\boxed{
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}
}
$$

This creates a mathematically meaningful interval containing the unknown global
optimal value.

### Global Optimality Gap

Define the certified global optimality gap as

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
}
$$

Then

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Therefore, $\Delta_{\mathrm{global}}$ is a valid upper bound on the difference
between the best value found by ARRGO and the true global optimum.

### Epsilon-Optimality

If

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Hence,

$$
\boxed{
f_{\mathrm{best}}
\text{ is an }
\varepsilon\text{-optimal objective value}
}
$$

under the validity of the upper-bound construction.

This is stronger than simply reporting the best value observed during the
search.

### Certification Requirement

The optimality-gap guarantee depends critically on the validity of the upper
envelopes.

In particular,

$$
P_{\mathrm{global}}
$$

is a certified upper bound only when the assumptions used to construct
$U_R(x)$ are valid.

For the Lipschitz-based version of ARRGO, this requires a valid global
Lipschitz constant.

Therefore,

$$
\boxed{
\text{Valid Upper Bound}
\Longrightarrow
\text{Valid Optimality Gap}
}
$$

whereas an empirical or heuristic upper estimate does not by itself provide a
rigorous certificate.

### Interpretation for ARRGO

The optimization potential answers a different question from uncertainty.

Uncertainty asks:

$$
\boxed{
\text{How much is still unknown?}
}
$$

Optimization potential asks:

$$
\boxed{
\text{How good could this region still be?}
}
$$

ARRGO uses both concepts because a region may have high uncertainty without
being particularly relevant to the global optimum, or may have high
optimization potential and therefore require further refinement.

Thus,

$$
\boxed{
\text{Uncertainty}
\neq
\text{Optimization Potential}
}
$$

and both are required for a complete certified global optimization analysis.

## Global Optimality Gap

The global optimality gap measures the maximum possible difference between the
best objective value currently found by ARRGO and the true global optimum.

Let

$$
f_{\mathrm{best}}
=
\max_{x_i\in\mathcal D} f(x_i)
$$

denote the best value obtained from all evaluated points.

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

denote the true global optimal value.

Since the evaluated points are contained in the search domain,

$$
f_{\mathrm{best}}\leq f^*.
$$

In certified mode, ARRGO maintains a valid global upper bound

$$
P_{\mathrm{global}}
\geq
f^*.
$$

Therefore,

$$
\boxed{
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}
}
$$

### Definition

The certified global optimality gap is defined as

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
}
$$

This quantity represents the largest remaining uncertainty about the
optimality of the incumbent that is justified by the current certified
information.

Because

$$
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}},
$$

we obtain

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Thus, the actual error of the incumbent is never larger than the certified
gap.

### Interpretation

The gap can be interpreted as the maximum amount by which the current
incumbent could still be improved according to the available certified
information.

A large value of

$$
\Delta_{\mathrm{global}}
$$

means that the current information does not yet provide a strong guarantee
about global optimality.

A small value means that the remaining possible improvement is tightly
bounded.

Therefore,

$$
\boxed{
\Delta_{\mathrm{global}}
\rightarrow 0
}
$$

represents increasing certainty about the global optimality of the incumbent.

### Epsilon-Optimality

For a prescribed tolerance $\varepsilon>0$, if

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Equivalently,

$$
\boxed{
f_{\mathrm{best}}
\geq
f^*-\varepsilon
}
$$

and therefore the incumbent is $\varepsilon$-optimal in objective value.

This provides a quantitative stopping criterion that is directly connected to
the quality of the optimization result.

### Gap Reduction Through Refinement

Suppose ARRGO performs a refinement operation that produces new information
without invalidating the existing upper bounds.

Let the global upper bound before refinement be

$$
P_{\mathrm{global}}^{(t)}
$$

and after refinement be

$$
P_{\mathrm{global}}^{(t+1)}.
$$

If the new information tightens the certified upper bound, then

$$
P_{\mathrm{global}}^{(t+1)}
\leq
P_{\mathrm{global}}^{(t)}.
$$

At the same time, the incumbent cannot become worse:

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

Therefore,

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}.
$$

Hence, under valid bound tightening,

$$
\boxed{
\Delta_{\mathrm{global}}
\text{ is non-increasing}
}
$$

through successful information refinement.

### Important Qualification

The monotonicity result depends on the upper bound being recomputed from
additional valid information.

It does not mean that every arbitrary sampling or splitting heuristic will
automatically reduce the gap.

ARRGO must therefore distinguish between:

$$
\boxed{
\text{Refinement}
}
$$

and

$$
\boxed{
\text{Effective Certified Refinement}
}
$$

The latter is refinement that preserves validity while improving the
information relevant to the global optimality bound.

### Relation to ARRGO Termination

The certified termination condition can be expressed as

$$
\boxed{
\Delta_{\mathrm{global}}\leq\varepsilon
}
$$

provided that all regional upper bounds contributing to
$P_{\mathrm{global}}$ are valid.

At that point, ARRGO does not merely return the best value observed. It can
return the incumbent together with a mathematical statement that its
objective value is within $\varepsilon$ of the true global optimum.

This distinction separates a certified optimization result from a
budget-limited best-found result.

## Epsilon-Optimality

A global optimization algorithm does not always need to identify the exact
global optimizer in order to provide a mathematically meaningful guarantee.

For a prescribed tolerance

$$
\varepsilon>0,
$$

an objective value is called $\varepsilon$-optimal if its distance from the
true global optimal value is no greater than $\varepsilon$.

### Definition

Let

$$
f^*
=
\max_{x\in\Omega}f(x)
$$

be the global optimal value, and let $\hat{x}\in\Omega$ be a candidate
solution.

The candidate $\hat{x}$ is $\varepsilon$-optimal in objective value if

$$
f^*-f(\hat{x})
\leq
\varepsilon.
$$

Equivalently,

$$
\boxed{
f(\hat{x})
\geq
f^*-\varepsilon
}
$$

This definition measures the quality of the objective value rather than the
distance between the candidate point and a particular global optimizer.

### Why Objective-Value Optimality Matters

A global optimization problem may have multiple global maximizers.

The set of global maximizers is

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

Therefore, requiring an algorithm to identify one specific optimizer is not
always necessary.

Moreover, two points can be far apart in the search domain while having
nearly identical objective values.

For this reason, ARRGO's primary certified accuracy measure is based on the
objective-value gap

$$
f^*-f(\hat{x}),
$$

rather than solely on the spatial distance

$$
|\hat{x}-x^*|.
$$

### ARRGO's Certified Condition

ARRGO maintains the incumbent

$$
x_{\mathrm{best}}
=
\operatorname*{arg\,max}_{x_i\in\mathcal D}f(x_i)
$$

with objective value

$$
f_{\mathrm{best}}
=
f(x_{\mathrm{best}}).
$$

In certified mode, suppose ARRGO maintains a valid global upper bound

$$
P_{\mathrm{global}}
\geq
f^*.
$$

The certified global optimality gap is

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

Since

$$
f^*
\leq
P_{\mathrm{global}},
$$

we have

$$
f^*-f_{\mathrm{best}}
\leq
P_{\mathrm{global}}-f_{\mathrm{best}}
=
\Delta_{\mathrm{global}}.
$$

Therefore, if

$$
\boxed{
\Delta_{\mathrm{global}}\leq\varepsilon
}
$$

then

$$
f^*-f_{\mathrm{best}}
\leq
\varepsilon.
$$

Hence,

$$
\boxed{
x_{\mathrm{best}}
\text{ is an }\varepsilon\text{-optimal solution}
}
$$

in objective value.

### Certified Versus Best-Found Results

It is important to distinguish two possible termination situations.

#### Certified Termination

If

$$
\Delta_{\mathrm{global}}\leq\varepsilon,
$$

and the upper bounds are mathematically valid, ARRGO can provide the
certificate

$$
f_{\mathrm{best}}
\geq
f^*-\varepsilon.
$$

This is a theoretical guarantee.

#### Budget-Based Termination

If ARRGO reaches its evaluation budget before the certified gap satisfies the
desired tolerance,

$$
N_t=N_{\max},
$$

the algorithm returns the best evaluated solution:

$$
x_{\mathrm{best}}.
$$

However, without a sufficiently small certified gap, the algorithm cannot
claim that this solution is $\varepsilon$-optimal.

Thus,

$$
\boxed{
\text{Best Found}
\neq
\text{Certified }\varepsilon\text{-Optimal}
}
$$

in general.

### Exact Optimality as a Special Case

If the certified gap reaches zero,

$$
\Delta_{\mathrm{global}}=0,
$$

then

$$
f_{\mathrm{best}}
=
f^*.
$$

Therefore, the incumbent achieves the exact global optimal objective value.

In practice, numerical computation generally uses a positive tolerance
$\varepsilon$, making $\varepsilon$-optimality the more practical stopping
criterion.

### Role in ARRGO

Epsilon-optimality connects the information maintained by ARRGO to a
quantitative statement about solution quality:

$$
\boxed{
\text{Certified Information}
\Longrightarrow
\text{Optimality Gap}
\Longrightarrow
\varepsilon\text{-Optimality}
}
$$

This establishes the mathematical meaning of ARRGO's certified termination
criterion.

## Convergence Conditions

The convergence analysis of ARRGO requires conditions that connect local
region refinement to the global optimization problem.

The objective is to establish that, under appropriate assumptions, the
algorithm cannot permanently exclude a region containing a global optimizer
and that the spatial resolution of relevant regions eventually becomes
arbitrarily fine.

The following conditions are required for the convergence analysis.

### Condition 1: Compact Search Domain

The search domain is a non-empty compact interval

$$
\Omega=[a,b],
\qquad a<b.
$$

Compactness guarantees the existence of a global maximizer when combined with
continuity of the objective function.

### Condition 2: Continuity of the Objective Function

The objective function satisfies

$$
f\in C(\Omega).
$$

Therefore, if a sequence of points converges,

$$
x_k\to x,
$$

then

$$
f(x_k)\to f(x).
$$

This condition allows increasingly small regions around a global optimizer to
represent increasingly precise information about its objective value.

### Condition 3: Valid Region Coverage

At every iteration, the collection of active and stable regions must preserve
coverage of the original search domain:

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R_t}R.
$$

For a non-overlapping partition, this becomes

$$
\Omega
=
\bigcup_{R\in\mathcal R_t}R.
$$

Thus, refinement changes the representation of the search space without
removing any part of the original domain.

### Condition 4: Valid Spatial Refinement

Whenever a region

$$
R=[l,r]
$$

is structurally refined at a split point $s$, its children

$$
R_L=[l,s],
\qquad
R_R=[s,r]
$$

must satisfy

$$
\max\{s-l,r-s\}
\leq
\rho(r-l),
\qquad
0<\rho<1.
$$

Consequently, repeated refinement along a nested sequence of regions produces

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0)
\to 0.
$$

### Condition 5: Information Refinement

Spatial refinement alone is not sufficient for ARRGO's information-driven
decision process.

The algorithm must also continue acquiring or propagating relevant
information about regions that remain potentially important.

In particular, a region containing a global optimizer must not remain
permanently unresolved merely because other regions repeatedly receive
refinement.

This condition connects the local refinement mechanism to the global
selection mechanism.

### Condition 6: Global Selection Fairness

Let $\mathcal R_t^{\mathrm{relevant}}$ denote regions whose current
information does not allow them to be safely excluded from containing a
global optimizer.

The global selection mechanism must satisfy the fairness requirement that
such a region cannot be ignored indefinitely.

Informally,

$$
R\in\mathcal R_t^{\mathrm{relevant}}
\quad\Longrightarrow\quad
R\text{ receives refinement opportunities over time}.
$$

This does not require every region to be refined at every iteration.

Instead, it prevents permanent starvation of regions that remain globally
relevant.

### Condition 7: Monotonic Incumbent Improvement

The incumbent objective value is defined by

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x_i\in\mathcal D_t}f(x_i),
$$

where $\mathcal D_t$ is the set of evaluated points available at iteration
$t$.

Because evaluations are retained,

$$
\mathcal D_t
\subseteq
\mathcal D_{t+1},
$$

and therefore

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

Thus, the best-found objective value cannot decrease as the algorithm
progresses.

### Condition 8: Deterministic Evaluation

The function evaluation is deterministic:

$$
f(x)
$$

returns the same value whenever the same point $x$ is evaluated.

Therefore, the information collected by ARRGO is reproducible and does not
require probabilistic convergence arguments.

### Condition 9: Valid Certified Bounds

For certified convergence and $\varepsilon$-optimality, ARRGO additionally
requires a valid upper-bounding mechanism.

In Certified Mode, if a valid Lipschitz constant $L$ is available,

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega,
$$

then the regional upper envelope satisfies

$$
f(x)\leq U_R(x),
\qquad
\forall x\in R.
$$

Consequently,

$$
\max_{x\in R}f(x)
\leq
P(R)
=
\max_{x\in R}U_R(x).
$$

These bounds provide the mathematical basis for determining whether the
remaining unexplored potential of the search space is sufficiently small.

### Combined Convergence Requirement

The central convergence structure of ARRGO can therefore be expressed as

$$
\boxed{
\text{Coverage}
+
\text{Valid Refinement}
+
\text{Information Refinement}
+
\text{Fair Global Selection}
}
$$

together with continuity of the objective function.

For certified convergence, valid global bounds are additionally required:

$$
\boxed{
\text{Convergence Structure}
+
\text{Valid Bounds}
\Longrightarrow
\text{Certified Accuracy}
}
$$

These conditions separate the roles of the different components of ARRGO.

Spatial contraction controls geometric resolution.

Information refinement controls what the algorithm learns about each region.

Global selection controls where refinement is allocated.

Valid bounds provide the connection between finite observations and a
mathematical statement about the unexplored optimum.

The next step is to combine these conditions into a formal convergence
statement.

## Convergence Theorem

We now combine the convergence conditions into a formal statement about the
spatial behavior of ARRGO.

### Theorem: Spatial Convergence of Relevant Refinement

Let

$$
\Omega=[a,b]
$$

be a compact search domain, and let

$$
f\in C(\Omega).
$$

Assume that ARRGO satisfies the following conditions:

1. The collection of regions maintained by ARRGO preserves coverage of
   $\Omega$.

2. Every structural refinement satisfies the contraction condition

   $$
   \max\{\operatorname{diam}(R_L),
   \operatorname{diam}(R_R)\}
   \leq
   \rho\,\operatorname{diam}(R),
   \qquad
   0<\rho<1.
   $$

3. A region that remains globally relevant cannot be permanently ignored by
   the global selection mechanism.

4. Relevant regions continue to receive valid refinement opportunities.

Then, for every nested sequence of relevant regions

$$
R_0\supseteq R_1\supseteq R_2\supseteq\cdots,
$$

generated through repeated structural refinement,

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Consequently,

$$
\boxed{
\lim_{k\to\infty}\operatorname{diam}(R_k)=0
}
$$

### Proof

By the contraction condition, each refinement satisfies

$$
\operatorname{diam}(R_{k+1})
\leq
\rho\operatorname{diam}(R_k).
$$

Applying this inequality recursively gives

$$
\operatorname{diam}(R_k)
\leq
\rho^k\operatorname{diam}(R_0).
$$

Because

$$
0<\rho<1,
$$

we have

$$
\lim_{k\to\infty}\rho^k=0.
$$

Therefore,

$$
\lim_{k\to\infty}
\operatorname{diam}(R_k)
=
0.
$$

Hence, repeated valid refinement of a relevant nested region produces
arbitrarily fine spatial resolution.

### Consequence for the Objective Function

Since

$$
f\in C(\Omega),
$$

continuity implies that sufficiently small spatial neighborhoods around a
point produce arbitrarily small changes in the objective value.

For every $\eta>0$ and every $x\in\Omega$, there exists
$\delta>0$ such that

$$
|x-y|<\delta
\quad\Longrightarrow\quad
|f(x)-f(y)|<\eta.
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

the spatial resolution of the relevant region can eventually become smaller
than any prescribed $\delta$.

Therefore, along a repeatedly refined relevant region, the possible
variation of the continuous objective becomes arbitrarily localized.

### Important Interpretation

This theorem establishes **spatial convergence of the refinement process**.

It does not, by itself, establish that the incumbent solution converges to a
global optimizer.

In particular,

$$
\operatorname{diam}(R_k)\to0
$$

does not imply

$$
x_{\mathrm{best}}^{(k)}\to x^*.
$$

To establish convergence of the optimization result, we additionally need
to connect:

$$
\boxed{
\text{Relevant Region Refinement}
\rightarrow
\text{Information Acquisition}
\rightarrow
\text{Global Optimization}
}
$$

This distinction is essential because an algorithm can refine regions
geometrically without necessarily evaluating sufficiently informative points
inside them.

### Certified Extension

If, in addition, $f$ satisfies a valid Lipschitz condition

$$
|f(x)-f(y)|
\leq
L|x-y|,
$$

then for any region $R_k$,

$$
\max_{x,y\in R_k}|f(x)-f(y)|
\leq
L\operatorname{diam}(R_k).
$$

Since

$$
\operatorname{diam}(R_k)\to0,
$$

we obtain

$$
L\operatorname{diam}(R_k)\to0.
$$

Thus, under the Lipschitz assumption, spatial contraction directly implies
that the maximum possible objective variation inside the repeatedly refined
region converges to zero.

This provides the mathematical bridge between spatial refinement and
certified uncertainty reduction.

The stronger statement that the global incumbent becomes
$\varepsilon$-optimal requires the global selection, bound validity, and
termination arguments established separately above.

## Global Convergence

Spatial convergence of a single nested region is not sufficient to establish
global convergence.

ARRGO is a global optimization framework, so its convergence mechanism must
also account for the entire collection of regions representing the search
domain.

Let

$$
\mathcal R_t
$$

denote the collection of regions maintained by ARRGO at iteration $t$.

The regions satisfy the coverage condition

$$
\Omega
\subseteq
\bigcup_{R\in\mathcal R_t}R.
$$

Therefore, every point in the original search domain remains represented by
at least one region.

### Global Relevance

Let

$$
R^*
$$

be a region that contains at least one global optimizer:

$$
R^*\cap X^*\neq\varnothing,
$$

where

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x).
$$

For ARRGO to converge globally, the refinement mechanism must not permanently
ignore $R^*$ while refining other regions.

This is the role of the global selection fairness condition.

### Fair Global Selection

Suppose that a region containing a global optimizer remains globally
relevant according to the information maintained by ARRGO.

Then the selection mechanism must provide infinitely many refinement
opportunities to that region unless its information becomes sufficient to
resolve its relevance.

In other words, there cannot exist a permanently relevant region $R^*$ and a
finite iteration $T$ such that

$$
R^*
\text{ receives no further refinement after }T.
$$

This prevents permanent starvation of potentially optimal regions.

### Global Refinement Structure

The global convergence mechanism can therefore be represented as

$$
\boxed{
\text{Coverage}
+
\text{Fair Selection}
+
\text{Contraction}
}
$$

where:

- **Coverage** ensures that no part of $\Omega$ disappears from the global
  representation.
- **Fair Selection** ensures that globally relevant regions cannot be ignored
  forever.
- **Contraction** ensures that repeatedly refined regions obtain arbitrarily
  fine spatial resolution.

Together, these properties prevent the algorithm from permanently focusing on
an isolated portion of the search domain without giving relevant competing
regions refinement opportunities.

### Relation to a Global Optimizer

Consider a global optimizer

$$
x^*\in X^*.
$$

At every iteration, because the region collection preserves coverage, there
exists at least one region

$$
R_t^*
$$

such that

$$
x^*\in R_t^*.
$$

If this region remains globally relevant and continues to receive refinement,
then along a corresponding nested sequence

$$
R_0^*
\supseteq
R_1^*
\supseteq
R_2^*
\supseteq
\cdots
$$

we obtain

$$
\operatorname{diam}(R_t^*)\to0.
$$

Therefore, the spatial representation of a globally optimal location can be
made arbitrarily precise.

### From Spatial Convergence to Solution Convergence

The remaining question is whether this increasingly precise representation
causes ARRGO to obtain increasingly good objective values.

Continuity gives

$$
x_t\to x^*
\quad\Longrightarrow\quad
f(x_t)\to f(x^*).
$$

Thus, if ARRGO generates evaluated points

$$
x_t\in R_t^*
$$

that converge to a global optimizer,

$$
x_t\to x^*,
$$

then

$$
f(x_t)\to f(x^*)=f^*.
$$

Since the incumbent satisfies

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_t),
$$

it follows that the sequence of best-found objective values approaches the
global optimum from below:

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

### Important Additional Requirement

The previous conclusion requires more than simply refining $R_t^*$.

The algorithm must actually generate evaluated points whose distance from the
relevant optimizer tends to zero.

Therefore, a complete convergence proof requires an additional
**evaluation-density or information-acquisition condition**.

A sufficient form is:

$$
\forall x^*\in X^*,
\qquad
\exists\{x_t\}
\text{ evaluated by ARRGO such that }
x_t\to x^*.
$$

Under this condition and continuity of $f$,

$$
f(x_t)\to f^*.
$$

Because the incumbent always retains the best evaluated value,

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_t),
$$

and because

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*,
$$

we obtain

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\to
f^*
}
$$

as the number of refinement opportunities tends to infinity.

### Global Convergence Principle

The global convergence logic of ARRGO can therefore be summarized as

$$
\boxed{
\begin{aligned}
&\text{Global Coverage}
\\
&\Downarrow
\\
&\text{Fair Selection of Relevant Regions}
\\
&\Downarrow
\\
&\text{Repeated Spatial Contraction}
\\
&\Downarrow
\\
&\text{Increasingly Precise Representation}
\\
&\Downarrow
\\
&\text{Evaluation Near Global Optimizers}
\\
&\Downarrow
\\
&f_{\mathrm{best}}^{(t)}\to f^*
\end{aligned}
}
$$

This establishes the conceptual structure required for global convergence.

A fully formal convergence theorem must additionally specify the exact
candidate-generation and action-selection rules that guarantee the required
evaluation-density condition.

Therefore, the implementation of ARRGO must preserve these theoretical
properties rather than relying only on heuristic region selection.

## Evaluation Density Condition

Spatial refinement guarantees that relevant regions can become arbitrarily
small. However, shrinking a region does not by itself guarantee that ARRGO
evaluates informative points sufficiently close to a global optimizer.

Therefore, the convergence analysis requires an additional condition on the
locations of function evaluations.

### Definition

Let

$$
\mathcal D_\infty
$$

denote the set of all points evaluated by ARRGO during an unbounded sequence
of iterations.

The evaluation set is said to satisfy the **global evaluation-density
condition** with respect to the global optimizer set $X^*$ if

$$
\forall x^*\in X^*,
\qquad
\inf_{x\in\mathcal D_\infty}|x-x^*|=0.
$$

Equivalently, for every global optimizer $x^*$ and every $\delta>0$, there
exists an evaluated point $x\in\mathcal D_\infty$ such that

$$
|x-x^*|<\delta.
$$

Thus, evaluated points occur arbitrarily close to every global optimizer.

### Why This Condition Is Necessary

Suppose ARRGO repeatedly refines a region containing a global optimizer
$x^*$.

It is possible, in principle, for the algorithm to reduce the diameter of
that region while repeatedly evaluating points that remain away from $x^*$.

Therefore,

$$
\operatorname{diam}(R_t)\to0
$$

alone does not imply

$$
\exists\,x_t\in\mathcal D_\infty:
x_t\to x^*.
$$

The evaluation-density condition explicitly rules out this failure mode.

### Connection with Continuity

Let

$$
x^*\in X^*
$$

and suppose the evaluation-density condition holds.

Then there exists a sequence of evaluated points

$$
x_k\in\mathcal D_\infty
$$

such that

$$
x_k\to x^*.
$$

Since

$$
f\in C(\Omega),
$$

continuity gives

$$
f(x_k)\to f(x^*).
$$

Because $x^*$ is a global optimizer,

$$
f(x^*)=f^*.
$$

Therefore,

$$
\boxed{
f(x_k)\to f^*
}
$$

as the evaluated points approach the global optimizer.

### Consequence for the Incumbent

The ARRGO incumbent is defined by

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x\in\mathcal D_t}f(x).
$$

Since every evaluated point is retained,

$$
\mathcal D_t
\subseteq
\mathcal D_{t+1},
$$

and therefore

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

For the sequence $x_k\to x^*$ described above,

$$
f(x_k)\to f^*.
$$

Since the incumbent is at least as good as every previously evaluated
point,

$$
f_{\mathrm{best}}^{(t_k)}
\geq
f(x_k)
$$

at the corresponding iterations $t_k$.

At the same time,

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*
$$

for every iteration.

Hence,

$$
\boxed{
\lim_{t\to\infty}f_{\mathrm{best}}^{(t)}
=
f^*
}
$$

provided that the global evaluation-density condition holds.

### Relation to Region Refinement

The evaluation-density condition can be achieved through the interaction of
three mechanisms:

$$
\boxed{
\text{Fair Region Selection}
+
\text{Spatial Contraction}
+
\text{Informative Evaluation}
}
$$

Fair selection prevents a globally relevant region from being permanently
ignored.

Spatial contraction makes the region containing a global optimizer
arbitrarily small.

Informative evaluation ensures that function evaluations are actually
generated sufficiently close to the optimizer.

### Certified Interpretation

In Certified Mode, the evaluation-density condition is not the only route to
a practical stopping guarantee.

If ARRGO obtains a valid global upper bound satisfying

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}-f_{\mathrm{best}}
\leq
\varepsilon,
$$

then $\varepsilon$-optimality follows directly from the certified gap,
regardless of whether the exact optimizer location has been identified.

Thus, two different theoretical goals must be distinguished:

$$
\boxed{
\text{Asymptotic Convergence}
}
$$

requires increasingly informative evaluations near global optimizers, whereas

$$
\boxed{
\text{Finite-Time Certified Accuracy}
}
$$

can be established through a sufficiently small valid global optimality gap.

### Design Requirement for ARRGO

The implementation must therefore ensure that its candidate-generation and
selection mechanisms are compatible with the evaluation-density condition.

The condition should not be assumed merely because regions are repeatedly
split.

It must emerge from the actual refinement policy used by ARRGO.

This requirement will later be used when formalizing the exact sampling and
splitting rules.

## Informative Evaluation Condition

The evaluation-density condition establishes that ARRGO must eventually
evaluate points arbitrarily close to global optimizers.

However, evaluation density alone does not describe how evaluation points are
selected.

ARRGO is an information-driven framework. Therefore, its sampling mechanism
must prefer points that provide useful information about the unresolved state
of a region.

### Information State of a Region

For a region $R$, define its information state as

$$
S_R
=
(D_R,B_R,Q_R),
$$

where:

- $D_R$ represents the observed function data,
- $B_R$ represents the observed local behavior,
- $Q_R$ represents the unresolved information profile.

The unresolved profile may contain components such as

$$
Q_R
=
[
Q_{\mathrm{coverage}},
Q_{\mathrm{behavior}},
Q_{\mathrm{uncertainty}},
Q_{\mathrm{potential}}
].
$$

A sampling action modifies the available information:

$$
S_R
\longrightarrow
S_R'.
$$

The purpose of informative sampling is to make this transition useful for
the subsequent optimization decision.

### Definition of Informative Sampling

Let $x\in R$ be a candidate sampling point.

The point is considered **informative** if evaluating $f(x)$ can provide
information that materially changes the representation, analysis, or
refinement decision associated with $R$.

This may occur through:

$$
\text{new spatial coverage},
$$

or

$$
\text{new behavioral information},
$$

or

$$
\text{reduction of certified uncertainty},
$$

or

$$
\text{improved estimation of optimization potential}.
$$

Therefore, informativeness is not equivalent to spatial novelty alone.

### Information Gain Without a Probability Model

ARRGO does not assume a probability distribution over the unknown function.

Consequently, the term "information gain" is interpreted deterministically.

For a candidate point $x$, define the information state before evaluation as

$$
S_R(x^-)
$$

and the state after evaluating the point as

$$
S_R(x^+).
$$

The resulting change is

$$
\Delta S_R(x)
=
S_R(x^+)-S_R(x^-).
$$

Because the components of $S_R$ may have different meanings and scales,
ARRGO does not combine them using arbitrary fixed numerical weights.

Instead, candidate points can be compared using their induced changes across
the relevant objectives.

### Pareto-Based Candidate Evaluation

Let the relevant information objectives for candidate $x$ be represented by

$$
G(x)
=
\left(
G_{\mathrm{coverage}}(x),
G_{\mathrm{behavior}}(x),
G_{\mathrm{uncertainty}}(x),
G_{\mathrm{potential}}(x)
\right).
$$

Candidate $x_1$ dominates candidate $x_2$ if

$$
G_i(x_1)\geq G_i(x_2)
$$

for every relevant objective $i$, and

$$
G_j(x_1)>G_j(x_2)
$$

for at least one objective $j$.

The set of non-dominated candidates forms the Pareto candidate set:

$$
\mathcal P_R
=
\left\{
x\in C_R:
\nexists\,z\in C_R
\text{ such that }z\text{ dominates }x
\right\},
$$

where $C_R$ is the set of candidate sampling locations.

This prevents ARRGO from introducing arbitrary weights merely to collapse
different information objectives into a single scalar score.

### Optimization Relevance

A candidate point may be informative but have little relevance to the global
optimization problem.

For example, additional samples in a region that is already known to have
low optimization potential may provide useful local information without
meaningfully affecting the global search.

Therefore, sampling decisions must consider both:

$$
\text{Information Value}
$$

and

$$
\text{Optimization Relevance}.
$$

Conceptually,

$$
\boxed{
\text{Informative Sampling}
=
\text{Information Value}
+
\text{Optimization Relevance}
}
$$

This expression describes the decision principle rather than a numerical
weighted sum.

### Relation to Convergence

The informative evaluation condition supports convergence by ensuring that
refinement is not merely geometric.

For a globally relevant region $R^*$, repeated refinement should produce
candidate evaluations that improve the information available about the
region.

When combined with fair global selection and spatial contraction, this
supports the required condition

$$
\exists\{x_k\}\subseteq\mathcal D_\infty
\quad\text{such that}\quad
x_k\to x^*,
$$

for a global optimizer $x^*$.

### Certified Mode

In Certified Mode, informative sampling has an additional interpretation.

A new evaluation can modify the lower and upper envelopes:

$$
L_R(x)
=
\max_i
\left[
f(x_i)-L|x-x_i|
\right],
$$

and

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

Consequently, a strategically selected evaluation point may reduce the
remaining certified uncertainty or reduce the region's optimization
potential.

Thus, in Certified Mode, sampling can be evaluated according to its ability
to tighten the mathematically valid description of the unexplored region.

### Design Principle

ARRGO should therefore not interpret sampling as simply "adding another
point."

Instead,

$$
\boxed{
\text{Sampling}
=
\text{Acquisition of Optimization-Relevant Information}
}
$$

The exact candidate-generation and comparison rules will be defined later
in the algorithm-design stage.

For the convergence analysis, the essential requirement is that the resulting
sampling policy must be capable of generating evaluations arbitrarily close
to globally optimal locations when those locations remain unresolved and
globally relevant.

## Limitations of the Convergence Guarantees

The convergence conditions established above provide a framework for analyzing
ARRGO, but they do not imply unrestricted global convergence under arbitrary
implementation choices.

The guarantees depend on the assumptions and properties explicitly stated in
the theoretical model.

### 1. Continuity Is Not Sufficient for Finite-Time Optimality

Continuity guarantees the existence of a global optimum on a compact domain
and supports convergence of objective values when evaluated points approach a
global optimizer.

However, continuity alone does not provide a numerical bound on the value of
an unseen point.

Therefore,

$$
f\in C(\Omega)
$$

does not by itself imply a finite-time certificate of the form

$$
f^*-f_{\mathrm{best}}\leq\varepsilon.
$$

A valid bounding assumption, such as a known Lipschitz constant, is required
for such a certificate.

### 2. Spatial Contraction Is Not Sufficient for Optimization Convergence

The property

$$
\operatorname{diam}(R_k)\to0
$$

only establishes increasing spatial resolution.

It does not guarantee that ARRGO evaluates points approaching the global
optimizer.

Therefore,

$$
\text{Spatial Contraction}
\not\Rightarrow
\text{Global Optimality}.
$$

The evaluation-density condition is required to connect spatial refinement
with objective-value convergence.

### 3. Sampling Is Not Automatically Informative

Adding function evaluations does not necessarily improve the optimization
decision.

A sampling point may:

- duplicate existing information,
- occur in an already well-resolved region,
- provide little behavioral information,
- or have little relevance to the remaining global potential.

Therefore, the sampling policy must explicitly account for the unresolved
information state of the region.

### 4. Heuristic Estimates Are Not Certified Bounds

ARRGO may estimate quantities such as local variation, slope behavior, or
candidate optimization potential from observed samples.

Such estimates are useful for guiding the search, but they are not
automatically mathematical bounds on the unknown function.

In particular,

$$
\widehat L_R
$$

is not necessarily a valid Lipschitz constant.

Therefore,

$$
\text{Estimated Bound}
\neq
\text{Certified Bound}
$$

unless its validity has been established independently.

Only valid upper bounds may be used to claim certified
$\varepsilon$-optimality.

### 5. Fair Selection Must Be Preserved

The global convergence argument requires that a region containing a global
optimizer cannot remain permanently ignored while it remains relevant.

If the implementation allows a globally relevant region to receive no
refinement opportunities indefinitely, the evaluation-density condition may
fail.

Consequently, the theoretical convergence argument depends on the actual
global selection policy rather than on the region representation alone.

### 6. Evaluation Budget Limits Asymptotic Guarantees

The convergence statements above concern an increasing number of refinement
opportunities.

A finite evaluation budget,

$$
N_{\max}<\infty,
$$

may terminate ARRGO before the required refinement level is reached.

In that case, ARRGO returns the best solution found within the available
budget:

$$
x_{\mathrm{best}}
=
\operatorname*{arg\,max}_{x_i\in\mathcal D}f(x_i).
$$

Unless a valid certified gap satisfies

$$
\Delta_{\mathrm{global}}\leq\varepsilon,
$$

the returned solution should not be described as a certified
$\varepsilon$-optimal solution.

### 7. Objective-Value Convergence Does Not Imply Location Convergence

Suppose

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

This establishes convergence of the best-found objective value.

It does not necessarily imply that the sequence of incumbent locations
converges to a unique point $x^*$.

This is particularly important when multiple global maximizers exist:

$$
|X^*|>1.
$$

Even when the optimal objective value is identified exactly, the optimizer
location may not be unique.

Therefore, ARRGO distinguishes between:

$$
\boxed{
\text{Objective-Value Convergence}
}
$$

and

$$
\boxed{
\text{Optimizer-Location Convergence}
}.
$$

The former is the primary convergence target of the current theoretical
framework.

### 8. No Universal Finite-Sample Guarantee Without Additional Structure

For a general continuous black-box function, a finite number of function
evaluations cannot, by itself, provide a non-trivial universal guarantee
about every unseen point.

Additional structural assumptions are required to construct rigorous
finite-sample bounds.

For ARRGO, the principal certified assumption is the known Lipschitz
condition

$$
|f(x)-f(y)|
\leq
L|x-y|.
$$

This assumption allows finite observations to produce valid envelopes over
unobserved parts of the search domain.

### Scope of the Current Guarantees

The theoretical framework therefore distinguishes three levels of claims.

#### Empirical Search Performance

ARRGO may demonstrate good optimization performance experimentally without
requiring a formal convergence certificate.

#### Asymptotic Convergence

Under the stated coverage, refinement, selection, and evaluation-density
conditions,

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

#### Certified Finite-Time Accuracy

Under a valid bounding assumption and a certified global gap,

$$
\Delta_{\mathrm{global}}
\leq
\varepsilon,
$$

ARRGO can guarantee

$$
f_{\mathrm{best}}
\geq
f^*-\varepsilon.
$$

These three levels must not be conflated.

The strongest claims require the strongest assumptions.

### Central Principle

The theoretical scope of ARRGO can therefore be summarized as

$$
\boxed{
\text{Assumptions}
\rightarrow
\text{Valid Algorithmic Properties}
\rightarrow
\text{Convergence or Certificate}
}
$$

rather than

$$
\boxed{
\text{Heuristic Behavior}
\rightarrow
\text{Automatic Guarantee}.
}
$$

This separation is fundamental to maintaining mathematical correctness in the
analysis of ARRGO.

## Certified Convergence

Certified convergence strengthens the asymptotic convergence framework by
introducing mathematically valid bounds on the unexplored objective values.

The certified analysis applies only when ARRGO has access to a valid
bounding assumption, such as a known Lipschitz constant.

### Certified Setting

Assume that

$$
\Omega=[a,b],
$$

and

$$
f\in C(\Omega).
$$

Additionally, assume that a valid constant $L\geq0$ is known such that

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

For every region

$$
R\subseteq\Omega,
$$

ARRGO maintains a set of evaluated samples

$$
D_R
=
\{(x_i,f(x_i))\}_{i=1}^{n_R}.
$$

### Valid Upper Envelope

For every evaluated point $x_i\in R$, the Lipschitz condition implies

$$
f(x)
\leq
f(x_i)+L|x-x_i|,
\qquad
\forall x\in R.
$$

Taking the minimum over all available samples gives the upper envelope

$$
U_R(x)
=
\min_i
\left[
f(x_i)+L|x-x_i|
\right].
$$

Therefore,

$$
\boxed{
f(x)\leq U_R(x)
}
$$

for every $x\in R$.

The regional optimization potential is defined as

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Consequently,

$$
\max_{x\in R}f(x)
\leq
P(R).
$$

Thus, $P(R)$ is a valid upper bound on the best objective value that can
still occur inside region $R$.

### Global Certified Potential

Suppose the current region collection preserves coverage of $\Omega$.

The global potential is

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal R_t}P(R).
$$

Because every point of $\Omega$ belongs to at least one maintained region,

$$
f^*
=
\max_{x\in\Omega}f(x)
\leq
P_{\mathrm{global}}.
$$

The incumbent satisfies

$$
f_{\mathrm{best}}
\leq
f^*.
$$

Therefore,

$$
\boxed{
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}
}
$$

at every certified iteration.

### Certified Global Gap

Define

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
$$

From the previous inequalities,

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Hence, the certified gap is an upper bound on the actual optimization error.

### Finite-Time Certificate

Let

$$
\varepsilon>0
$$

be the desired objective-value tolerance.

If ARRGO reaches an iteration $t$ satisfying

$$
\Delta_{\mathrm{global}}^{(t)}
\leq
\varepsilon,
$$

then

$$
f^*
-
f_{\mathrm{best}}^{(t)}
\leq
\varepsilon.
$$

Therefore,

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\geq
f^*-\varepsilon
}
$$

and the current incumbent is certified to be $\varepsilon$-optimal in
objective value.

This provides a finite-time stopping certificate.

### Why Refinement Can Improve the Certificate

A valid new function evaluation adds information to the regional data set.

For a fixed region $R$, if

$$
D_R^{(t)}
\subseteq
D_R^{(t+1)},
$$

then the corresponding upper envelopes satisfy

$$
U_R^{(t+1)}(x)
\leq
U_R^{(t)}(x),
\qquad
\forall x\in R.
$$

Therefore,

$$
P^{(t+1)}(R)
\leq
P^{(t)}(R).
$$

At the same time, the incumbent cannot decrease:

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

Hence, whenever refinement adds valid information,

$$
\boxed{
\Delta_{\mathrm{global}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}
}
$$

provided that the global region representation continues to use valid
upper bounds.

Thus, certified refinement produces a monotonically non-increasing upper
bound on the remaining optimization error.

### Contraction and Certified Resolution

Consider a region $R$ with diameter

$$
d_R=\operatorname{diam}(R).
$$

Under the Lipschitz condition, for any $x,y\in R$,

$$
|f(x)-f(y)|
\leq
Ld_R.
$$

Therefore,

$$
\boxed{
\operatorname{osc}_R(f)
\leq
Ld_R
}
$$

where

$$
\operatorname{osc}_R(f)
=
\max_{x,y\in R}|f(x)-f(y)|.
$$

If repeated structural refinement satisfies

$$
d_R\to0,
$$

then

$$
Ld_R\to0.
$$

Consequently, the maximum possible variation of the objective inside the
refined region converges to zero.

This establishes the fundamental connection

$$
\boxed{
\text{Spatial Contraction}
\Longrightarrow
\text{Vanishing Lipschitz Variation}
}
$$

### Certified Convergence Principle

Certified convergence in ARRGO therefore relies on two complementary
mechanisms.

The first is **information tightening**:

$$
D_R^{(t+1)}
\supseteq
D_R^{(t)}
\quad\Longrightarrow\quad
P(R) \text{ does not increase}.
$$

The second is **spatial contraction**:

$$
\operatorname{diam}(R_t)\to0
\quad\Longrightarrow\quad
L\operatorname{diam}(R_t)\to0.
$$

Together with global coverage and fair refinement of relevant regions, these
properties allow the certified global potential to approach the quality of
the incumbent.

The final finite-time guarantee is therefore

$$
\boxed{
\Delta_{\mathrm{global}}
\leq
\varepsilon
\Longrightarrow
x_{\mathrm{best}}
\text{ is }\varepsilon\text{-optimal}.
}
$$

This certificate is conditional on the validity of the assumed Lipschitz
bound and on the correctness of the upper-bound computation.

It does not claim that the exact global optimizer location has been
identified.

## Monotonicity of the Certified Global Gap

The certified global optimality gap is one of the central quantities maintained
by ARRGO in Certified Mode.

It measures the maximum remaining difference that is consistent with the
current certified information.

Recall that

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

A desirable property of a certified optimization framework is that acquiring
additional valid information should not increase this gap.

### Proposition

Assume that between iterations $t$ and $t+1$:

1. all previously valid function evaluations are retained,
2. all newly computed upper bounds remain valid,
3. the maintained regions continue to cover $\Omega$.

Then

$$
\boxed{
\Delta_{\mathrm{global}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}
}
$$

provided that the global potential is recomputed from the updated valid
regional bounds.

### Proof

Let the regional data at iteration $t$ be

$$
D_R^{(t)}.
$$

After additional valid evaluations,

$$
D_R^{(t)}
\subseteq
D_R^{(t+1)}.
$$

The upper envelope at iteration $t$ is

$$
U_R^{(t)}(x)
=
\min_{x_i\in D_R^{(t)}}
\left[
f(x_i)+L|x-x_i|
\right].
$$

At iteration $t+1$, the minimum is taken over a superset of the previous
sample set:

$$
U_R^{(t+1)}(x)
=
\min_{x_i\in D_R^{(t+1)}}
\left[
f(x_i)+L|x-x_i|
\right].
$$

Adding elements to a minimization set cannot increase its minimum.
Therefore,

$$
U_R^{(t+1)}(x)
\leq
U_R^{(t)}(x),
\qquad
\forall x\in R.
$$

Taking the maximum over the region gives

$$
P^{(t+1)}(R)
=
\max_{x\in R}U_R^{(t+1)}(x)
\leq
\max_{x\in R}U_R^{(t)}(x)
=
P^{(t)}(R).
$$

Therefore, additional valid observations cannot increase the certified
potential of a fixed region.

### Effect of the Incumbent

Because all previous evaluations are retained,

$$
\mathcal D_t
\subseteq
\mathcal D_{t+1}.
$$

The incumbent is therefore monotonic:

$$
f_{\mathrm{best}}^{(t+1)}
=
\max_{x\in\mathcal D_{t+1}}f(x)
\geq
\max_{x\in\mathcal D_t}f(x)
=
f_{\mathrm{best}}^{(t)}.
$$

Thus, the lower side of the certified interval cannot decrease.

### Global Gap

The global potential is

$$
P_{\mathrm{global}}^{(t)}
=
\max_{R\in\mathcal R_t}P^{(t)}(R).
$$

When the region representation is refined, the children preserve the
corresponding portion of the search domain and valid bounds are recomputed
for them.

Because refinement does not remove valid information or enlarge the true
set of possible objective values, the resulting global certified potential
cannot increase:

$$
P_{\mathrm{global}}^{(t+1)}
\leq
P_{\mathrm{global}}^{(t)}.
$$

Together with

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)},
$$

we obtain

$$
\begin{aligned}
\Delta_{\mathrm{global}}^{(t+1)}
&=
P_{\mathrm{global}}^{(t+1)}
-
f_{\mathrm{best}}^{(t+1)}
\\
&\leq
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}
\\
&=
\Delta_{\mathrm{global}}^{(t)}.
\end{aligned}
$$

Hence,

$$
\boxed{
\Delta_{\mathrm{global}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}
}
$$

which proves the proposition.

### Interpretation

The certified gap behaves monotonically:

$$
\Delta_{\mathrm{global}}^{(0)}
\geq
\Delta_{\mathrm{global}}^{(1)}
\geq
\Delta_{\mathrm{global}}^{(2)}
\geq
\cdots
\geq0.
$$

Therefore, once ARRGO reaches a certified tolerance,

$$
\Delta_{\mathrm{global}}^{(t)}
\leq
\varepsilon,
$$

later valid refinement cannot invalidate that certificate.

This gives the termination criterion a stable mathematical interpretation.

### Important Qualification

Monotonicity does not imply that the gap decreases strictly at every
iteration.

It is possible that

$$
\Delta_{\mathrm{global}}^{(t+1)}
=
\Delta_{\mathrm{global}}^{(t)}.
$$

For example, a new evaluation may provide information in a region that is
not currently responsible for the global potential.

Therefore,

$$
\boxed{
\text{Non-Increasing}
\neq
\text{Strictly Decreasing}
}
$$

The algorithm may require multiple refinement actions before the globally
relevant upper bound changes.

### Relation to ARRGO's Design

This property reinforces an important design rule:

ARRGO must retain valid information rather than discarding observations or
replacing valid bounds with weaker estimates.

The certified state should therefore evolve according to

$$
\boxed{
\text{More Valid Information}
\Longrightarrow
\text{No Worse Certificate}
}
$$

This monotonicity property will later be used when defining the certified
termination rule and the implementation of the global region priority.

## Vanishing Certified Gap

Monotonicity of the certified global gap guarantees that the gap cannot
increase when valid information is accumulated.

However, monotonicity alone does not imply convergence to zero.

A non-increasing sequence may converge to a positive value. Therefore, an
additional condition is required to establish that the certified gap can
become arbitrarily small.

### Certified Gap Sequence

Let

$$
\Delta_t
=
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

From the monotonicity property,

$$
0
\leq
\Delta_{t+1}
\leq
\Delta_t.
$$

Therefore, the sequence $\{\Delta_t\}$ is bounded below by zero and
non-increasing.

By the monotone convergence theorem for real sequences, there exists a limit
$\Delta_\infty\geq0$ such that

$$
\lim_{t\to\infty}\Delta_t
=
\Delta_\infty.
$$

The remaining question is whether

$$
\Delta_\infty=0.
$$

### Sufficient Refinement Condition

A sufficient condition for vanishing certified gap is that every region
which can contain a global optimizer receives arbitrarily fine refinement and
sufficient function information.

In particular, let $R_t^*$ be a sequence of regions containing a global
optimizer $x^*$.

Assume that

$$
\operatorname{diam}(R_t^*)
\to0
$$

and that the certified upper bound over this sequence becomes tight:

$$
P(R_t^*)
-
\max_{x\in R_t^*}f(x)
\to0.
$$

If the incumbent simultaneously satisfies

$$
f_{\mathrm{best}}^{(t)}
\to f^*,
$$

then

$$
P_{\mathrm{global}}^{(t)}
\to f^*
$$

and consequently

$$
\boxed{
\Delta_{\mathrm{global}}^{(t)}
\to0.
}
$$

### Why the Relevant Region Is Sufficient

Because $R_t^*$ contains a global optimizer,

$$
x^*\in R_t^*,
$$

we have

$$
\max_{x\in R_t^*}f(x)
=
f^*.
$$

The global potential satisfies

$$
P_{\mathrm{global}}^{(t)}
\geq
P(R_t^*)
\geq
f^*.
$$

If the upper bound over the relevant region becomes tight,

$$
P(R_t^*)\to f^*,
$$

then the contribution of that region to the global certified potential
approaches the true optimum.

At the same time, because the incumbent approaches the optimum,

$$
f_{\mathrm{best}}^{(t)}\to f^*.
$$

Therefore, the difference between the global upper potential and the
incumbent approaches zero.

### Lipschitz-Based Sufficient Condition

Under a valid Lipschitz constant $L$, spatial contraction provides a direct
mechanism for making the unresolved variation inside a region small.

For a region $R_t^*$,

$$
\operatorname{osc}_{R_t^*}(f)
\leq
L\operatorname{diam}(R_t^*).
$$

If

$$
\operatorname{diam}(R_t^*)\to0,
$$

then

$$
L\operatorname{diam}(R_t^*)\to0.
$$

Therefore, the maximum possible variation of the objective inside the
relevant region vanishes.

This gives a sufficient mechanism for tightening the regional certified
description, provided that the upper-bound construction retains valid
samples and the region remains properly represented after refinement.

### Connection to Evaluation Density

Spatial contraction alone is not enough to guarantee

$$
f_{\mathrm{best}}^{(t)}\to f^*.
$$

The evaluation-density condition provides the missing link.

If evaluated points approach a global optimizer,

$$
x_t\to x^*,
$$

then continuity implies

$$
f(x_t)\to f^*.
$$

Since the incumbent retains the best evaluated value,

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_t),
$$

and

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*,
$$

we obtain

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

Thus, the two sides of the certified gap are controlled by complementary
mechanisms:

$$
\boxed{
\begin{aligned}
P_{\mathrm{global}}^{(t)}
&\to f^*
&&\text{through valid bound tightening},\\
f_{\mathrm{best}}^{(t)}
&\to f^*
&&\text{through informative evaluation}.
\end{aligned}
}
$$

Consequently,

$$
\boxed{
\Delta_{\mathrm{global}}^{(t)}
\to0.
}
$$

### Important Distinction

The statement

$$
\Delta_{\mathrm{global}}^{(t)}
\text{ is non-increasing}
$$

is weaker than

$$
\Delta_{\mathrm{global}}^{(t)}
\to0.
$$

The first follows from retaining valid information and a monotonic
incumbent.

The second additionally requires sufficient refinement and information
acquisition in globally relevant regions.

Therefore, the theoretical convergence claim for ARRGO depends on the full
combination:

$$
\boxed{
\text{Coverage}
+
\text{Fair Selection}
+
\text{Spatial Contraction}
+
\text{Evaluation Density}
+
\text{Valid Bounds}
}
$$

rather than on monotonicity of the gap alone.

### Consequence for Certified Termination

If

$$
\Delta_{\mathrm{global}}^{(t)}
\to0,
$$

then for every prescribed

$$
\varepsilon>0,
$$

there exists a finite iteration $T$ such that

$$
t\geq T
\quad\Longrightarrow\quad
\Delta_{\mathrm{global}}^{(t)}
\leq
\varepsilon.
$$

At that point ARRGO can terminate with the certified guarantee

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\geq
f^*-\varepsilon.
}
$$

Thus, vanishing of the certified global gap provides the theoretical basis
for finite-time $\varepsilon$-optimal termination whenever the required
conditions are satisfied.

## Global Objective-Value Convergence Theorem

The previous sections established the individual properties required for
global convergence.

We now combine these properties into a single theorem concerning the
objective value returned by ARRGO.

### Theorem

Let

$$
\Omega=[a,b]
$$

be a compact search domain, and let

$$
f\in C(\Omega).
$$

Assume that ARRGO satisfies the following conditions:

1. **Global Coverage**

   The maintained region collection preserves the representation of the
   entire search domain:

   $$
   \Omega
   \subseteq
   \bigcup_{R\in\mathcal R_t}R.
   $$

2. **Valid Spatial Contraction**

   Every structural refinement satisfies

   $$
   \operatorname{diam}(R_{k+1})
   \leq
   \rho\operatorname{diam}(R_k),
   \qquad
   0<\rho<1.
   $$

3. **Fair Global Selection**

   A region containing a global optimizer and remaining globally relevant
   cannot be permanently ignored.

4. **Evaluation Density**

   For every global optimizer

   $$
   x^*\in X^*,
   $$

   ARRGO generates evaluated points arbitrarily close to $x^*$:

   $$
   \inf_{x\in\mathcal D_\infty}|x-x^*|=0.
   $$

5. **Persistent Evaluation History**

   Previously evaluated points are retained, so

   $$
   \mathcal D_t
   \subseteq
   \mathcal D_{t+1}.
   $$

Then the sequence of ARRGO's best-found objective values satisfies

$$
\boxed{
\lim_{t\to\infty}
f_{\mathrm{best}}^{(t)}
=
f^*
}
$$

where

$$
f^*
=
\max_{x\in\Omega}f(x).
$$

### Proof

Because the evaluation-density condition holds, for any global optimizer
$x^*\in X^*$ there exists a sequence of evaluated points

$$
x_k\in\mathcal D_\infty
$$

such that

$$
x_k\to x^*.
$$

Since $f$ is continuous,

$$
f(x_k)\to f(x^*).
$$

Because $x^*$ is a global optimizer,

$$
f(x^*)=f^*.
$$

Therefore,

$$
f(x_k)\to f^*.
$$

Now consider the incumbent objective value

$$
f_{\mathrm{best}}^{(t)}
=
\max_{x\in\mathcal D_t}f(x).
$$

Since every evaluated point remains available,

$$
f_{\mathrm{best}}^{(t)}
\geq
f(x_k)
$$

for every sufficiently large iteration corresponding to the evaluation of
$x_k$.

At the same time, because $f^*$ is the maximum of $f$ over $\Omega$,

$$
f_{\mathrm{best}}^{(t)}
\leq
f^*.
$$

Thus, along the sequence of iterations associated with the evaluations
$x_k$,

$$
f(x_k)
\leq
f_{\mathrm{best}}^{(t_k)}
\leq
f^*.
$$

Since

$$
f(x_k)\to f^*,
$$

the squeeze theorem gives

$$
f_{\mathrm{best}}^{(t_k)}
\to
f^*.
$$

Furthermore, the incumbent sequence is monotone non-decreasing:

$$
f_{\mathrm{best}}^{(t+1)}
\geq
f_{\mathrm{best}}^{(t)}.
$$

A monotone non-decreasing sequence bounded above by $f^*$ has a limit.
Because a subsequence converges to $f^*$, the full sequence must have the
same limit.

Therefore,

$$
\boxed{
\lim_{t\to\infty}
f_{\mathrm{best}}^{(t)}
=
f^*
}.
$$

This proves global convergence in objective value.

### Interpretation

The theorem does not require ARRGO to identify a unique global optimizer.

If several points satisfy

$$
f(x)=f^*,
$$

the algorithm only needs to generate evaluations arbitrarily close to at
least one global optimizer for the objective-value convergence result.

Therefore, the theorem establishes

$$
\boxed{
\text{Global Objective-Value Convergence}
}
$$

rather than necessarily

$$
\boxed{
\text{Convergence of the Optimizer Location}
}.
$$

### Certified Extension

If a valid Lipschitz constant is additionally available, ARRGO can maintain
valid global upper bounds and define

$$
\Delta_{\mathrm{global}}^{(t)}
=
P_{\mathrm{global}}^{(t)}
-
f_{\mathrm{best}}^{(t)}.
$$

If the certified gap satisfies

$$
\Delta_{\mathrm{global}}^{(t)}
\to0,
$$

then for every

$$
\varepsilon>0
$$

there exists a finite iteration $T$ such that

$$
\Delta_{\mathrm{global}}^{(T)}
\leq
\varepsilon.
$$

Consequently,

$$
f_{\mathrm{best}}^{(T)}
\geq
f^*-\varepsilon.
$$

Thus, Certified ARRGO provides not only asymptotic objective-value
convergence, but also a finite-time $\varepsilon$-optimality certificate
whenever the certified gap reaches the requested tolerance.

### Scope of the Theorem

The theorem is conditional on the stated assumptions.

In particular, it does not claim convergence for an arbitrary heuristic
implementation of ARRGO.

The actual implementation must ensure that its region-selection,
refinement, and candidate-generation mechanisms satisfy the properties
required by the theorem.

Therefore, the theoretical result can be summarized as

$$
\boxed{
\begin{aligned}
&\text{Coverage}
+
\text{Fair Selection}
+
\text{Contraction}
+
\text{Evaluation Density}
\\
&\qquad\qquad\Downarrow
\\
&f_{\mathrm{best}}^{(t)}\to f^*
\end{aligned}
}
$$

and, with valid certified bounds,

$$
\boxed{
\Delta_{\mathrm{global}}\leq\varepsilon
\Longrightarrow
\varepsilon\text{-optimality}.
}
$$

## Optimizer-Location Convergence

The global convergence theorem establishes convergence of the best-found
objective value:

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

However, convergence of the objective value does not, in general, imply
convergence of the corresponding optimizer location.

### Objective-Value Convergence

The primary convergence result established for ARRGO is

$$
\boxed{
f_{\mathrm{best}}^{(t)}
\to
f^*
}.
$$

This means that the quality of the best-found solution approaches the global
optimal objective value.

It does not necessarily imply that

$$
x_{\mathrm{best}}^{(t)}
\to
x^*
$$

for a particular global optimizer $x^*$.

### Multiple Global Optimizers

Suppose that the global optimizer set is

$$
X^*
=
\operatorname*{arg\,max}_{x\in\Omega}f(x)
$$

and contains multiple points.

For example,

$$
x_1^*,x_2^*\in X^*,
\qquad
x_1^*\neq x_2^*.
$$

Then

$$
f(x_1^*)
=
f(x_2^*)
=
f^*.
$$

An algorithm may therefore approach different members of $X^*$ while
achieving the same optimal objective value.

Consequently, the existence of

$$
f_{\mathrm{best}}^{(t)}
\to
f^*
$$

does not determine a unique limiting optimizer location.

### Unique Optimizer Is Still Not Sufficient by Itself

Even if the global optimizer is unique,

$$
X^*=\{x^*\},
$$

objective-value convergence alone does not automatically provide a complete
proof of location convergence.

Additional information about the local structure of the objective near
$x^*$ is required.

A suitable condition is a local growth condition stating that sufficiently
large spatial deviations from $x^*$ necessarily produce a measurable loss in
objective value.

### Local Growth Condition

Suppose there exist constants

$$
c>0
\qquad\text{and}\qquad
p>0
$$

and a neighborhood of $x^*$ such that

$$
f^*-f(x)
\geq
c|x-x^*|^p.
$$

Then, for a candidate point $\hat{x}$,

$$
f^*-f(\hat{x})
\geq
c|\hat{x}-x^*|^p.
$$

Rearranging gives

$$
|\hat{x}-x^*|
\leq
\left(
\frac{f^*-f(\hat{x})}{c}
\right)^{1/p}.
$$

Therefore, if

$$
f(\hat{x}_t)\to f^*,
$$

then

$$
|\hat{x}_t-x^*|
\to0.
$$

Hence,

$$
\boxed{
f(\hat{x}_t)\to f^*
+
\text{Local Growth Condition}
\Longrightarrow
\hat{x}_t\to x^*
}.
$$

### Interpretation for ARRGO

This distinction leads to two different theoretical objectives.

The first is **objective-value convergence**:

$$
f_{\mathrm{best}}^{(t)}
\to
f^*.
$$

The second is **optimizer-location convergence**:

$$
x_{\mathrm{best}}^{(t)}
\to
x^*.
$$

The first requires continuity together with the global refinement and
evaluation conditions established previously.

The second requires additional structural assumptions on the objective,
such as uniqueness and a suitable local growth condition.

### Certified Accuracy Versus Location Accuracy

The certified gap

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}
$$

provides a guarantee on objective-value accuracy:

$$
\Delta_{\mathrm{global}}\leq\varepsilon
\Longrightarrow
f_{\mathrm{best}}
\geq
f^*-\varepsilon.
$$

It does not directly provide a bound of the form

$$
|x_{\mathrm{best}}-x^*|
\leq
\delta.
$$

Such a spatial guarantee requires additional assumptions connecting
objective-value loss to spatial distance.

Therefore,

$$
\boxed{
\text{Objective-Value Certificate}
\neq
\text{Location Certificate}
}.
$$

### Role of ARRGO's Spatial Refinement

Although the primary theoretical guarantee concerns objective value,
ARRGO's spatial refinement mechanism still provides increasingly fine
localization of potentially optimal regions.

For a nested sequence of relevant regions,

$$
R_0^*
\supseteq
R_1^*
\supseteq
R_2^*
\supseteq
\cdots,
$$

we have

$$
\operatorname{diam}(R_t^*)\to0.
$$

Therefore, the spatial uncertainty associated with that region vanishes.

If the region contains a unique global optimizer $x^*$ and the refinement
sequence continues around that optimizer, the region itself provides an
increasingly precise localization of $x^*$.

This is a stronger statement than merely observing convergence of the
objective value, but it still depends on the actual refinement and selection
behavior of the implementation.

### Theoretical Scope

The current ARRGO framework therefore adopts the following hierarchy:

$$
\boxed{
\text{Global Objective-Value Convergence}
}
$$

is the primary general convergence guarantee.

Under additional structural assumptions,

$$
\boxed{
\text{Optimizer-Location Convergence}
}
$$

may also be established.

This separation prevents the theoretical analysis from making a stronger
claim than is justified by the assumptions.

### Summary

The distinction can be expressed as

$$
\boxed{
\begin{aligned}
f_{\mathrm{best}}^{(t)}\to f^*
&\quad\text{is an objective-value statement},\\
x_{\mathrm{best}}^{(t)}\to x^*
&\quad\text{is a location statement}.
\end{aligned}
}
$$

ARRGO's core theoretical framework guarantees the first under its stated
convergence conditions.

The second requires additional assumptions concerning the structure and
uniqueness of the global optimum.

## Consistency of Regional Bounds

In Certified Mode, ARRGO maintains valid upper bounds for individual
regions.

Because regions may be sampled, split, and refined repeatedly, the
corresponding bounds must remain consistent with the changing region
structure.

### Parent Region

Consider a parent region

$$
R=[l,r]
$$

with a valid upper bound

$$
f(x)\leq U_R(x),
\qquad
\forall x\in R.
$$

Its certified optimization potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\max_{x\in R}f(x)
\leq
P(R).
$$

### Structural Refinement

Suppose $R$ is split at a point $s$ into

$$
R_L=[l,s],
$$

and

$$
R_R=[s,r].
$$

The children satisfy

$$
R_L\cup R_R=R.
$$

Consequently,

$$
\max_{x\in R}f(x)
=
\max
\left\{
\max_{x\in R_L}f(x),
\max_{x\in R_R}f(x)
\right\}.
$$

This identity follows directly from the fact that the two child regions
together cover the parent region.

### Validity of Child Bounds

Let the certified upper bounds of the children be

$$
U_{R_L}(x)
$$

and

$$
U_{R_R}(x).
$$

They are valid if

$$
f(x)\leq U_{R_L}(x),
\qquad
\forall x\in R_L,
$$

and

$$
f(x)\leq U_{R_R}(x),
\qquad
\forall x\in R_R.
$$

Therefore,

$$
\max_{x\in R_L}f(x)
\leq
P(R_L),
$$

and

$$
\max_{x\in R_R}f(x)
\leq
P(R_R).
$$

Hence,

$$
\max_{x\in R}f(x)
\leq
\max\{P(R_L),P(R_R)\}.
$$

The global potential of the refined representation therefore remains a
valid upper bound.

### Information Inheritance

Samples collected in the parent region remain valid observations after
splitting.

Suppose

$$
D_R
=
\{(x_i,f(x_i))\}.
$$

For a child region $R_c$, the inherited sample set is

$$
D_{R_c}^{\mathrm{inherit}}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_c\}.
$$

These observations remain valid because the function values themselves have
not changed.

New samples can then be added independently:

$$
D_{R_c}
=
D_{R_c}^{\mathrm{inherit}}
\cup
D_{R_c}^{\mathrm{new}}.
$$

Thus, structural refinement changes the organization of the information but
does not invalidate previously evaluated function values.

### Lipschitz-Based Child Bounds

Under a valid Lipschitz constant $L$, each inherited sample $(x_i,f(x_i))$
provides the bound

$$
f(x)
\leq
f(x_i)+L|x-x_i|.
$$

Therefore, the child upper envelope can be constructed as

$$
U_{R_c}(x)
=
\min_{x_i\in D_{R_c}}
\left[
f(x_i)+L|x-x_i|
\right],
\qquad
x\in R_c.
$$

Because every term in the minimum is a valid upper bound,

$$
f(x)\leq U_{R_c}(x).
$$

Thus, recomputing the envelope using inherited and newly evaluated samples
preserves certified validity.

### Parent Bound Versus Child Bounds

The parent bound and child bounds play different roles.

The parent bound describes the possible objective values over

$$
R.
$$

After splitting, the child bounds describe the corresponding possibilities
over

$$
R_L
\quad\text{and}\quad
R_R.
$$

Because

$$
R_L\cup R_R=R,
$$

the child representation preserves the search space.

The parent region therefore does not need to remain the sole source of the
certified bound after structural refinement.

However, the parent itself remains part of ARRGO's hierarchy and historical
representation.

### No Pruning

ARRGO does not delete regions from its global hierarchy.

After splitting,

$$
R
\longrightarrow
\{R_L,R_R\},
$$

the parent remains represented together with its descendants.

This distinction is important:

$$
\boxed{
\text{Hierarchical Retention}
\neq
\text{Active Search Priority}
}
$$

A parent region may cease to be an active refinement target while still
remaining available as part of the search history and structural hierarchy.

### Global Consistency

At every certified iteration, ARRGO must maintain the invariant

$$
\boxed{
f(x)
\leq
P(R)
\quad
\text{for the appropriate maintained region }R
}
$$

for every point represented by the region structure.

Consequently,

$$
f^*
\leq
P_{\mathrm{global}}.
$$

Therefore, splitting and information inheritance do not change the
mathematical meaning of the global certified potential.

### Certified Refinement Invariant

The complete invariant can be expressed as

$$
\boxed{
\begin{aligned}
&\text{Valid Samples}
\\
&\Downarrow
\\
&\text{Valid Regional Bounds}
\\
&\Downarrow
\\
&\text{Valid Regional Potentials}
\\
&\Downarrow
\\
&\text{Valid Global Potential}
\\
&\Downarrow
\\
&f^*\leq P_{\mathrm{global}}.
\end{aligned}
}
$$

This invariant must hold after every sampling and splitting operation.

It provides the mathematical foundation required for ARRGO to preserve
certified correctness throughout the entire refinement process.

## Region Hierarchy and Inherited Information

ARRGO represents the search space as a hierarchical collection of regions.

Each refinement operation may transform one region into smaller child regions
while preserving the original region as part of the search history.

### Region Tree

Let the initial search region be

$$
R_0.
$$

After refinement, a region may be divided into child regions:

$$
R
\longrightarrow
\{R_1,R_2,\ldots,R_k\}.
$$

Repeated refinement produces a hierarchy that can be represented as a tree.

Each node corresponds to a region, and each child represents a structural
refinement of its parent.

For every child $R_c$ of a parent $R_p$,

$$
R_c\subseteq R_p.
$$

The hierarchy therefore represents increasingly localized portions of the
original search domain.

### Region Identity

A region is not defined only by its current interval.

ARRGO conceptually associates each region with:

$$
R=
(\text{geometry},
\text{samples},
\text{behavior},
\text{bounds},
\text{state},
\text{parent},
\text{children}).
$$

The geometric component describes where the region lies.

The information components describe what ARRGO currently knows about that
region.

The structural components describe how the region was created and how it is
related to other regions.

### Inherited Samples

Suppose a parent region contains the evaluated set

$$
D_R.
$$

After splitting, each child receives the observations that lie inside its
domain.

For child $R_c$,

$$
D_{R_c}^{\mathrm{inherit}}
=
\{(x_i,f(x_i))\in D_R:x_i\in R_c\}.
$$

These observations do not need to be evaluated again.

Thus, splitting changes the spatial organization of the information without
discarding previously acquired function evaluations.

### Global Evaluation History

ARRGO also maintains a global evaluation history

$$
D_t
=
\{(x_i,f(x_i))\}_{i=1}^{N_t}.
$$

A region-specific dataset can therefore be viewed as a projection of the
global history onto that region:

$$
D_R
=
\{(x_i,f(x_i))\in D_t:x_i\in R\}.
$$

This separation is useful because the same evaluated point may become
relevant to different structural interpretations as the hierarchy evolves.

### Inherited Behavioral Information

Observed behavior may also be inherited when it remains valid for the child.

For example, if neighboring samples from the parent provide information
about local variation, those observations remain available to the child.

However, ARRGO must distinguish between:

$$
\text{Observed Information}
$$

and

$$
\text{Assumed Information}.
$$

Only information directly supported by valid observations or mathematical
assumptions may be used in certified reasoning.

A behavioral interpretation derived from the parent must therefore be
recomputed or restricted when the child requires a more localized analysis.

### Inherited Certified Information

Under the Lipschitz assumption, every inherited sample remains a valid source
of a certified bound.

For a sample $(x_i,f(x_i))$,

$$
f(x)
\leq
f(x_i)+L|x-x_i|.
$$

Therefore, restricting the domain from $R_p$ to a child $R_c$ does not
invalidate the inequality.

The same sample can continue contributing to the child's upper envelope:

$$
U_{R_c}(x)
=
\min_{x_i\in D_{R_c}}
\left[
f(x_i)+L|x-x_i|
\right].
$$

Consequently, previously acquired certified information can be reused rather
than recomputed from scratch.

### Structural Refinement Does Not Destroy History

When

$$
R_p\rightarrow\{R_1,R_2,\ldots,R_k\},
$$

ARRGO preserves:

1. the parent region,
2. its evaluated observations,
3. its structural metadata,
4. its relationship with the generated children.

The children additionally acquire their own localized state.

Therefore, refinement is an information-expanding operation rather than a
replacement operation.

### Parent-Child Consistency

For every child $R_c$,

$$
R_c\subseteq R_p.
$$

If the children form a complete partition of the parent, then

$$
\bigcup_c R_c=R_p.
$$

This provides structural consistency between levels of the hierarchy.

The hierarchy must therefore satisfy:

$$
\boxed{
\text{Child Geometry}
\subseteq
\text{Parent Geometry}
}
$$

and, for a complete split,

$$
\boxed{
\bigcup_c R_c=R_p.
}
$$

### Information Monotonicity

ARRGO should not lose valid evaluated information as refinement proceeds.

If

$$
D_t\subseteq D_{t+1},
$$

then every previously evaluated point remains available to the algorithm.

This gives the information-history invariant

$$
\boxed{
D_t\subseteq D_{t+1}.
}
$$

New refinement steps may add information, but they must not invalidate
previously verified observations.

### Local Relevance

Although global history is persistent, not every observation is equally
relevant to every region.

ARRGO therefore distinguishes between:

$$
\text{Global Information}
$$

and

$$
\text{Region-Relevant Information}.
$$

Global information can influence priorities and comparisons between regions.

Region-relevant information is used for local behavioral analysis,
uncertainty estimation, and refinement decisions.

This prevents the algorithm from treating distant observations as if they
were automatically equivalent to local observations.

### Hierarchical Search State

At iteration $t$, the region hierarchy can be represented as

$$
\mathcal{H}_t
=
\{R_0,R_1,\ldots,R_m\},
$$

together with the parent-child relationships between its elements.

The hierarchy satisfies:

$$
\boxed{
\text{Persistence}
+
\text{Containment}
+
\text{Information Inheritance}
}
$$

throughout refinement.

These properties allow ARRGO to maintain a persistent representation of the
search process while progressively increasing spatial and informational
resolution.

### Role in ARRGO

The region hierarchy provides three essential capabilities:

- **History preservation:** previously explored regions remain represented.
- **Information reuse:** valid samples and certified information can be
  inherited by descendants.
- **Progressive localization:** repeated refinement creates increasingly
  smaller regions around areas requiring additional information.

Therefore, the hierarchy is not merely a data structure.

It is part of the mathematical representation of ARRGO's refinement process.

## Refinement Completeness and Fairness

ARRGO relies on global region selection to determine which region receives
the next refinement operation.

Spatial contraction alone is not sufficient for global convergence.

A region containing a global optimizer could remain unresolved forever if the
selection mechanism permanently ignores it.

Therefore, ARRGO requires a fairness condition on global region selection.

### Global Selection

Let

$$
\mathcal{R}_t
$$

denote the set of regions represented in the global search structure at
iteration $t$.

At each iteration, ARRGO selects a region

$$
R_t\in\mathcal{R}_t
$$

for further analysis or refinement.

The selection mechanism is allowed to be adaptive and information-driven.

However, it must satisfy a fundamental fairness property.

### Fairness Condition

Consider a region $R$ whose unresolved optimization information remains
relevant to the global search.

ARRGO must not permanently exclude $R$ from future consideration.

Conceptually,

$$
R\text{ remains globally relevant}
\quad\Longrightarrow\quad
R\text{ receives future refinement opportunities}.
$$

This does not require every region to be refined at every iteration.

Instead, it prevents permanent starvation of a relevant region.

### Persistent Relevance

Define a region as persistently relevant if its available information continues
to indicate that it may contain a better solution than the current incumbent.

In Certified Mode, this can be expressed using the regional potential:

$$
P(R)>f_{\mathrm{best}}.
$$

More generally, for a tolerance $\epsilon$,

$$
P(R)>f_{\mathrm{best}}+\epsilon
$$

indicates that the region may still contain an improvement larger than the
current target accuracy.

Such a region cannot be permanently ignored if ARRGO is to establish a
global certificate.

### Refinement Completeness

ARRGO satisfies refinement completeness if every persistently relevant region
eventually receives sufficient refinement.

A conceptual form of this condition is:

$$
\boxed{
\text{Persistent Relevance}
\Longrightarrow
\text{Eventual Refinement}
}
$$

where sufficient refinement means that the unresolved information required
for the global decision is progressively reduced.

This condition does not prescribe one specific scheduling strategy.

It defines the property that the scheduling strategy must satisfy.

### Why Fairness Is Necessary

Suppose the search domain contains two regions:

$$
R_A
\quad\text{and}\quad
R_B.
$$

Assume the global optimizer lies inside $R_B$.

If ARRGO repeatedly refines only $R_A$, then

$$
\operatorname{diam}(R_A)\rightarrow0
$$

does not imply anything about $R_B$.

The optimizer-containing region may remain large and poorly resolved.

Therefore,

$$
\boxed{
\text{Contraction of Some Regions}
\not\Rightarrow
\text{Global Convergence}.
}
$$

The refinement process must reach every region that remains relevant to the
global optimization problem.

### Fairness and Global Selection

The fairness condition applies to the interaction between:

$$
\text{Region Priority}
$$

and

$$
\text{Region Selection}.
$$

A region may have lower priority at one iteration and higher priority later.

Therefore, ARRGO must recompute global priorities as new information becomes
available.

This allows information discovered in one region to change the relative
importance of other regions.

### Certified Fairness

In Certified Mode, fairness is particularly important because the global
certificate depends on the maximum regional potential:

$$
P_{\mathrm{global}}
=
\max_R P(R).
$$

If a region with a large potential is permanently ignored, then its bound may
remain unnecessarily loose.

Consequently, the global potential may fail to decrease sufficiently:

$$
P_{\mathrm{global}}^{(t)}
\not\rightarrow f^*.
$$

Fair refinement prevents such unresolved competitive regions from being
permanently excluded.

### Tolerance-Based Fairness

For a target tolerance $\epsilon>0$, ARRGO can define competitive regions by

$$
\mathcal{C}_t(\epsilon)
=
\left\{
R\in\mathcal{R}_t:
P(R)>f_{\mathrm{best}}^{(t)}+\epsilon
\right\}.
$$

These regions represent areas that are still capable, according to the valid
certificate, of improving the incumbent by more than $\epsilon$.

A sufficient fairness requirement is that regions that remain persistently
in

$$
\mathcal{C}_t(\epsilon)
$$

cannot be ignored indefinitely.

Their certified uncertainty or potential must eventually be reduced.

### Empirical Fairness Versus Certified Fairness

ARRGO distinguishes between two settings.

In Empirical Mode, fairness supports asymptotic search coverage and convergence
arguments but does not provide a finite-time optimality certificate.

In Certified Mode, fairness is combined with valid bounds and is used to ensure
that unresolved competitive regions are eventually resolved sufficiently for
the global certificate.

Therefore,

$$
\boxed{
\text{Fairness}
\neq
\text{Optimality Guarantee}
}
$$

but fairness is one of the conditions required for the guarantee.

### Interaction With Contraction

Fairness and contraction serve different purposes.

Contraction provides:

$$
\operatorname{diam}(R_k)\rightarrow0
$$

for regions that are repeatedly refined.

Fairness determines which globally relevant regions are guaranteed to
receive such refinement.

Together they provide:

$$
\boxed{
\text{Fair Selection}
+
\text{Spatial Contraction}
\Rightarrow
\text{Progressive Global Resolution}.
}
$$

Additional information-acquisition conditions are still required to establish
objective-value convergence.

### Interaction With Evaluation Density

Fair refinement does not automatically guarantee that the algorithm evaluates
points arbitrarily close to every global optimizer.

Therefore, ARRGO separately requires the evaluation-density condition:

$$
\forall x^*\in X^*,
\qquad
\inf_{x\in D_\infty}|x-x^*|=0.
$$

Fair region selection supports this property, but candidate-generation and
sampling rules must also ensure that refinement produces sufficiently
informative evaluations.

Thus,

$$
\boxed{
\text{Fair Region Selection}
+
\text{Informative Evaluation}
\Rightarrow
\text{Evaluation Density}.
}
$$

The exact sufficient conditions for this implication belong to the algorithm
design and candidate-generation layer.

### No Requirement for Uniform Refinement

ARRGO does not require all regions to have the same refinement depth.

For example,

$$
\operatorname{depth}(R_A)=10
$$

and

$$
\operatorname{depth}(R_B)=3
$$

may be completely valid if the available information justifies the
difference.

The requirement is not uniform refinement.

The requirement is that no persistently relevant region is permanently
starved.

Therefore,

$$
\boxed{
\text{Adaptive Refinement}
\neq
\text{Uniform Refinement}.
}
$$

### Refinement Completeness Invariant

The central invariant is:

$$
\boxed{
\text{A Persistently Competitive Region Cannot Be Permanently Ignored}.
}
$$

This invariant connects ARRGO's local refinement mechanism to its global
convergence properties.

Without it, local refinement could produce arbitrarily fine regions while the
global optimizer remains unresolved.

With it, global selection becomes a mathematically meaningful component of
the convergence framework.

### Role in ARRGO

Refinement completeness establishes the bridge between local and global
reasoning:

$$
\boxed{
\text{Local Contraction}
+
\text{Global Fairness}
+
\text{Informative Evaluation}
}
$$

forms the structural basis for ARRGO's global convergence mechanism.

The concrete implementation of fairness will be defined later when the global
region-priority and region-selection rules are formalized.

## Termination Correctness

Termination determines when ARRGO stops refining the search space and returns
its current incumbent.

A termination event must be distinguished from simple exhaustion of the
available computational budget.

### Two Termination Modes

ARRGO distinguishes between:

1. **Certified Termination**
2. **Budget-Limited Termination**

These two modes have different theoretical meanings.

### Certified Termination

In Certified Mode, ARRGO maintains a valid global optimality gap

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}-f_{\mathrm{best}}.
$$

Because

$$
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}},
$$

we obtain

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

Therefore, if

$$
\boxed{
\Delta_{\mathrm{global}}\leq\epsilon
}
$$

for a specified tolerance $\epsilon>0$, then

$$
f_{\mathrm{best}}
\geq
f^*-\epsilon.
$$

Hence, the incumbent is guaranteed to be $\epsilon$-optimal.

### Correctness of the Certificate

The termination condition is valid only if the global potential is itself
valid.

ARRGO must maintain

$$
f^*
\leq
P_{\mathrm{global}}.
$$

Therefore, certified termination requires all of the following:

- valid function evaluations,
- valid regional upper bounds,
- complete coverage of the search domain,
- valid regional potentials,
- correct aggregation of regional potentials,
- correct incumbent value.

If any of these conditions fail, the resulting gap cannot be interpreted as a
mathematical certificate.

### Budget-Limited Termination

ARRGO may also terminate because the evaluation budget has been exhausted.

Let

$$
N_t
$$

denote the number of function evaluations at iteration $t$, and let

$$
N_{\max}
$$

denote the available evaluation budget.

Budget termination occurs when

$$
N_t\geq N_{\max}.
$$

The algorithm then returns

$$
x_{\mathrm{best}}^{(t)}
=
\arg\max_{x_i\in D_t}f(x_i).
$$

This result represents the best solution found within the available budget.

It does not automatically imply

$$
f(x_{\mathrm{best}}^{(t)})=f^*.
$$

Therefore,

$$
\boxed{
\text{Budget Exhaustion}
\neq
\text{Global Optimality}.
}
$$

### Tolerance-Based Termination

A tolerance $\epsilon$ defines the maximum acceptable objective-value error in
Certified Mode.

The certified stopping rule is

$$
P_{\mathrm{global}}-f_{\mathrm{best}}
\leq
\epsilon.
$$

Equivalently,

$$
P_{\mathrm{global}}
\leq
f_{\mathrm{best}}+\epsilon.
$$

Since $f^*\leq P_{\mathrm{global}}$, it follows that

$$
f^*
\leq
f_{\mathrm{best}}+\epsilon.
$$

Thus,

$$
\boxed{
f_{\mathrm{best}}
\geq
f^*-\epsilon.
}
$$

### Zero-Tolerance Limit

If

$$
\epsilon=0,
$$

the certificate condition becomes

$$
P_{\mathrm{global}}=f_{\mathrm{best}}.
$$

Combined with

$$
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}},
$$

we obtain

$$
f_{\mathrm{best}}
=
f^*
=
P_{\mathrm{global}}.
$$

Thus, under exact arithmetic and valid certified bounds, zero global gap
corresponds to exact objective-value optimality.

In practical numerical computation, however, exact equality is replaced by
appropriate numerical tolerances.

### Termination and Unresolved Regions

In Certified Mode, ARRGO must not terminate merely because the currently
selected region appears locally stable.

A locally stable region does not necessarily imply global optimality.

For example, another region may still satisfy

$$
P(R)>f_{\mathrm{best}}+\epsilon.
$$

Such a region remains globally competitive.

Therefore, certified termination requires the global condition

$$
\forall R\in\mathcal{R}_t,
\qquad
P(R)\leq f_{\mathrm{best}}+\epsilon.
$$

Equivalently,

$$
\max_{R\in\mathcal{R}_t}P(R)
\leq
f_{\mathrm{best}}+\epsilon.
$$

This is stronger than checking only the region currently being refined.

### Termination After Refinement

Suppose ARRGO performs a refinement operation and obtains

$$
\Delta_{\mathrm{global}}^{(t+1)}
\leq
\Delta_{\mathrm{global}}^{(t)}.
$$

The algorithm must recompute the global state after the refinement.

Only then can the stopping condition be evaluated.

The correct sequence is:

$$
\boxed{
\text{Refine}
\rightarrow
\text{Update Information}
\rightarrow
\text{Update Bounds}
\rightarrow
\text{Update Incumbent}
\rightarrow
\text{Recompute Global Gap}
\rightarrow
\text{Check Termination}.
}
$$

This prevents termination decisions from being based on stale information.

### Termination Does Not Delete the Hierarchy

When ARRGO terminates, the region hierarchy remains conceptually available.

Termination means that no further refinement is required under the selected
stopping criterion.

It does not mean that previously explored regions are deleted.

Therefore,

$$
\boxed{
\text{Termination}
\neq
\text{Pruning of Search History}.
}
$$

### Correctness Statement

The certified termination rule can be summarized as follows.

**Proposition.**

Assume that:

1. the search domain is completely covered by maintained regions;
2. every regional upper bound is valid;
3. every regional potential is computed correctly;
4. the global potential is the maximum regional potential;
5. the incumbent is the maximum value among all evaluated points.

Then

$$
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}-f_{\mathrm{best}}
\leq
\epsilon
$$

implies

$$
\boxed{
f_{\mathrm{best}}
\geq
f^*-\epsilon.
}
$$

Therefore, ARRGO's certified termination criterion is sufficient for
$\epsilon$-optimality.

### Interpretation of Returned Results

ARRGO therefore reports its final result according to the termination mode.

For certified termination:

$$
\boxed{
\text{Returned solution is }\epsilon\text{-optimal}.
}
$$

For budget-limited termination:

$$
\boxed{
\text{Returned solution is the best solution found within the budget}.
}
$$

This distinction prevents ARRGO from making a stronger mathematical claim
than its available information supports.

### Termination Correctness Invariant

The central principle is:

$$
\boxed{
\text{ARRGO may claim certified optimality only when its valid global gap
satisfies the requested tolerance.}
}
$$

This makes termination a consequence of the mathematical state of the search,
rather than an arbitrary stopping decision.

## Numerical Precision and Tolerance

The theoretical formulation of ARRGO is expressed using real-valued
mathematics.

Actual implementation, however, uses finite-precision numerical
representation.

Therefore, numerical precision must be treated separately from the
mathematical properties of the optimization framework.

### Exact Mathematical Model

In the mathematical formulation, function evaluations are represented as
exact real values:

$$
y_i=f(x_i).
$$

Likewise, region boundaries and candidate locations are treated as exact
real numbers.

Under this model, equality and ordering operations have their usual
mathematical meaning.

### Finite-Precision Computation

In implementation, numerical values are represented using finite-precision
floating-point arithmetic.

An evaluated point is therefore represented by an approximation

$$
\hat{x}_i
$$

and its computed function value by

$$
\hat{y}_i.
$$

In general,

$$
\hat{x}_i\neq x_i
$$

and

$$
\hat{y}_i\neq y_i
$$

at the level of machine representation.

The implementation must therefore avoid relying on exact floating-point
equality when numerical quantities are expected to be approximately equal.

### Numerical Tolerance

ARRGO may use a numerical tolerance

$$
\tau>0
$$

to determine whether two numerical values should be treated as
indistinguishable for a specific computational operation.

For example, two points $x_1$ and $x_2$ may be considered numerically
duplicate when

$$
|x_1-x_2|\leq\tau.
$$

This is a computational rule and must not be confused with a mathematical
statement that

$$
x_1=x_2.
$$

### Separation of Tolerances

ARRGO should distinguish between different types of tolerance.

At minimum, the framework may require:

- **Point tolerance:** determines numerical duplication of evaluation
  locations.
- **Objective tolerance:** determines whether two function values are
  numerically indistinguishable.
- **Termination tolerance:** determines the requested optimization accuracy.
- **Boundary tolerance:** handles numerical comparisons with region
  boundaries.

These tolerances serve different purposes and should not automatically be
identified with one another.

### Point Duplication

A function evaluation should not be repeated unnecessarily at the same
numerical location.

For candidate point $x_c$ and an existing evaluated point $x_i$, ARRGO may
apply

$$
|x_c-x_i|\leq\tau_x
$$

as the duplicate criterion.

If the criterion is satisfied, the candidate is considered already
represented by the evaluation history.

This avoids wasting evaluation budget on numerically indistinguishable
points.

### Boundary Consistency

Suppose a region is

$$
R=[l,r].
$$

A candidate point intended to lie inside the region may be numerically
represented slightly outside the boundary because of floating-point
rounding.

The implementation must therefore apply a consistent boundary policy.

Conceptually, a candidate belongs to $R$ when it satisfies the numerical
membership rule associated with

$$
l\leq x\leq r.
$$

The same convention must be used when assigning evaluations to parent and
child regions.

### Split-Point Consistency

Suppose

$$
R=[l,r]
$$

is split at

$$
s.
$$

The resulting children are

$$
R_L=[l,s]
$$

and

$$
R_R=[s,r].
$$

The split point must be represented consistently so that the union of the
children preserves the parent region:

$$
R_L\cup R_R=R.
$$

A numerical implementation must avoid creating an artificial gap or
inconsistency at $s$.

### Ordering of Function Values

When comparing two computed objective values

$$
\hat{f}_1
\quad\text{and}\quad
\hat{f}_2,
$$

a numerical tolerance may be used when their difference is close to the
machine precision or another explicitly defined comparison scale.

However, tolerance-based comparison must not silently alter the mathematical
optimization objective.

The implementation should therefore distinguish between:

$$
\text{numerical indistinguishability}
$$

and

$$
\text{actual objective equality}.
$$

### Certified Bounds and Numerical Error

The theoretical certified bound

$$
f(x)\leq U_R(x)
$$

assumes that the quantities used to construct $U_R$ are mathematically
valid.

Floating-point rounding can introduce small numerical errors into the
computed representation.

Therefore, a practical certified implementation must account for numerical
rounding when evaluating or comparing certified bounds.

A computed value should not be treated as a mathematically rigorous
certificate merely because it is represented by a floating-point number.

### Certified Versus Empirical Numerical Results

ARRGO distinguishes between:

$$
\text{Mathematical Certificate}
$$

and

$$
\text{Numerical Approximation of a Certificate}.
$$

The mathematical guarantee depends on the validity of the underlying
assumptions and inequalities.

The implementation additionally requires sufficient numerical care so that
floating-point errors do not invalidate the intended comparisons.

Therefore, a production-level certified implementation may require
conservative numerical margins.

### Termination Tolerance

The optimization tolerance $\epsilon$ has a different interpretation from
machine precision.

The condition

$$
\Delta_{\mathrm{global}}\leq\epsilon
$$

means that the requested objective-value uncertainty is at most $\epsilon$.

It does not mean that the computation itself is accurate to exactly
$\epsilon$ in floating-point arithmetic.

Therefore,

$$
\boxed{
\epsilon_{\mathrm{optimization}}
\neq
\epsilon_{\mathrm{numerical}}.
}
$$

### Tolerance Hierarchy

A practical implementation should maintain a clear hierarchy between
numerical tolerances and optimization tolerances.

Conceptually,

$$
\tau_{\mathrm{machine}}
\ll
\tau_{\mathrm{numerical}}
\ll
\epsilon_{\mathrm{optimization}},
$$

whenever the selected problem scale makes such a hierarchy appropriate.

The exact numerical values must be determined by the scale and conditioning
of the problem rather than chosen as universal constants.

### Scale Dependence

Numerical tolerances should be interpreted relative to the scale of the
optimization problem.

For example, a point tolerance appropriate for a domain of size

$$
10^{-2}
$$

may be inappropriate for a domain of size

$$
10^6.
$$

Therefore, ARRGO should avoid assuming that one fixed absolute tolerance is
universally appropriate for every objective or search domain.

Relative and absolute criteria may be combined when required by the numerical
implementation.

### Numerical Stability of Refinement

Refinement must remain meaningful at small scales.

Suppose repeated splitting produces

$$
r-l\rightarrow0.
$$

Eventually, floating-point representation may make two theoretically
different points numerically indistinguishable.

At that point, further structural refinement may no longer provide meaningful
numerical resolution.

ARRGO must therefore recognize numerical resolution limits separately from
theoretical convergence.

### Numerical Stopping Condition

A practical implementation may terminate when further refinement cannot
produce a numerically distinguishable candidate or region.

This is not equivalent to certified $\epsilon$-optimality.

Thus,

$$
\boxed{
\text{Numerical Resolution Limit}
\neq
\text{Certified Optimality}.
}
$$

If both conditions are available, they should be reported separately.

### Theoretical Versus Implementational Guarantees

The theoretical analysis assumes exact mathematical operations.

The implementation operates under finite precision.

Therefore, ARRGO has two layers of correctness:

$$
\boxed{
\text{Mathematical Correctness}
}
$$

and

$$
\boxed{
\text{Numerical Implementation Correctness}.
}
$$

The first establishes what the algorithm guarantees under its mathematical
assumptions.

The second ensures that the implementation respects those assumptions as
closely as required by finite-precision computation.

### Numerical Precision Invariant

The implementation should preserve the following principle:

$$
\boxed{
\text{Numerical Tolerance Must Not Be Used to Invent Mathematical Guarantees}.
}
$$

Numerical tolerances are implementation mechanisms for robust computation.
They do not replace the assumptions required for convergence or certified
optimality.

This distinction allows ARRGO to remain mathematically rigorous while still
being implementable using standard floating-point numerical computation.

## Theoretical Summary and Assumption-to-Guarantee Map

The theoretical framework of ARRGO separates assumptions, algorithmic
conditions, intermediate properties, and final guarantees.

This separation is necessary because different guarantees require different
levels of assumptions.

### Core Mathematical Assumptions

ARRGO is defined over a compact search domain

$$
\Omega=[a,b],
\qquad
a<b.
$$

The objective function is deterministic and continuous:

$$
f:\Omega\rightarrow\mathbb{R},
\qquad
f\in C(\Omega).
$$

Therefore, a global optimum exists:

$$
\exists x^*\in\Omega:
\qquad
f(x^*)=f^*.
$$

### Structural Conditions

ARRGO maintains a collection of regions covering the search domain.

The region structure must preserve:

$$
\bigcup_{R\in\mathcal{R}_t}R=\Omega.
$$

Refinement must preserve valid geometric relationships between parent and
child regions.

For a refined region,

$$
R\rightarrow\{R_1,\ldots,R_k\},
$$

we require

$$
R_i\subseteq R
$$

and

$$
\bigcup_i R_i=R.
$$

### Contraction Condition

Repeated refinement must reduce the diameter of a repeatedly refined region.

For contraction factor

$$
0<\rho<1,
$$

the refinement operation satisfies

$$
\operatorname{diam}(R_{t+1})
\leq
\rho\operatorname{diam}(R_t).
$$

Therefore,

$$
\operatorname{diam}(R_t)
\leq
\rho^t\operatorname{diam}(R_0)
\rightarrow0.
$$

This establishes spatial resolution of repeatedly refined regions.

### Global Selection Condition

Spatial contraction is meaningful globally only if relevant regions are not
permanently ignored.

Therefore, ARRGO requires a fairness condition:

$$
\boxed{
\text{Persistent Global Relevance}
\Longrightarrow
\text{Eventual Refinement Opportunity}.
}
$$

This condition connects local refinement to global search.

### Information Acquisition Condition

Refinement must produce meaningful information.

ARRGO therefore requires that relevant regions receive evaluations or
structural refinements sufficient to improve the resolution of the
optimization problem.

In particular, for global optimizers,

$$
\forall x^*\in X^*,
\qquad
\inf_{x\in D_\infty}|x-x^*|=0.
$$

This is the evaluation-density condition.

### Objective-Value Convergence

Under:

1. compactness,
2. continuity,
3. global coverage,
4. valid contraction,
5. fair global selection,
6. evaluation density,
7. persistent evaluation history,

the best observed objective value satisfies

$$
\boxed{
\lim_{t\rightarrow\infty}
f_{\mathrm{best}}^{(t)}
=
f^*.
}
$$

This is an objective-value convergence guarantee.

It does not by itself imply convergence of the returned location to one
particular optimizer when multiple global optimizers exist.

### Additional Assumption for Certified Mode

Certified Mode additionally assumes a valid Lipschitz constant $L$ such that

$$
|f(x)-f(y)|
\leq
L|x-y|,
\qquad
\forall x,y\in\Omega.
$$

This assumption allows ARRGO to construct mathematically valid upper
envelopes.

For region $R$,

$$
U_R(x)
=
\min_{x_i\in D_R}
\left[
f(x_i)+L|x-x_i|
\right].
$$

Then

$$
f(x)\leq U_R(x).
$$

### Certified Regional Potential

The certified regional potential is

$$
P(R)
=
\max_{x\in R}U_R(x).
$$

Therefore,

$$
\max_{x\in R}f(x)
\leq
P(R).
$$

Aggregating all maintained regions gives

$$
P_{\mathrm{global}}
=
\max_{R\in\mathcal{R}_t}P(R).
$$

If the regions cover $\Omega$, then

$$
f^*
\leq
P_{\mathrm{global}}.
$$

### Certified Global Gap

The incumbent is

$$
f_{\mathrm{best}}
=
\max_{x_i\in D_t}f(x_i).
$$

Hence,

$$
f_{\mathrm{best}}
\leq
f^*
\leq
P_{\mathrm{global}}.
$$

The certified global gap is

$$
\boxed{
\Delta_{\mathrm{global}}
=
P_{\mathrm{global}}
-
f_{\mathrm{best}}.
}
$$

Consequently,

$$
0
\leq
f^*-f_{\mathrm{best}}
\leq
\Delta_{\mathrm{global}}.
$$

### Certified Finite-Time Guarantee

If

$$
\Delta_{\mathrm{global}}
\leq
\epsilon,
$$

then

$$
f_{\mathrm{best}}
\geq
f^*-\epsilon.
$$

Therefore,

$$
\boxed{
\Delta_{\mathrm{global}}\leq\epsilon
\Longrightarrow
\epsilon\text{-optimality}.
}
$$

This is a finite-time certificate.

It does not require knowing the exact location of the global optimizer.

### Assumption-to-Guarantee Map

The theoretical dependencies can be summarized as follows.

$$
\boxed{
\begin{array}{c}
\text{Compact Domain}
+
\text{Continuity}
\\[4pt]
\Downarrow
\\[4pt]
\text{Existence of Global Optimum}
\end{array}
}
$$

Spatial properties:

$$
\boxed{
\text{Valid Refinement}
+
\text{Contraction}
\Longrightarrow
\text{Spatial Resolution}
}
$$

Global properties:

$$
\boxed{
\text{Coverage}
+
\text{Fair Selection}
+
\text{Informative Evaluation}
\Longrightarrow
\text{Global Search Resolution}
}
$$

Objective-value convergence:

$$
\boxed{
\text{Global Search Resolution}
+
\text{Continuity}
\Longrightarrow
f_{\mathrm{best}}\rightarrow f^*
}
$$

Certified reasoning:

$$
\boxed{
\text{Valid Lipschitz Bound}
+
\text{Valid Regional Bounds}
\Longrightarrow
f^*\leq P_{\mathrm{global}}
}
$$

Certified optimality:

$$
\boxed{
f^*\leq P_{\mathrm{global}}
+
\Delta_{\mathrm{global}}\leq\epsilon
\Longrightarrow
\epsilon\text{-optimality}
}
$$

### Guarantee Hierarchy

ARRGO therefore has three distinct theoretical levels.

#### Level 1: Empirical Performance

With finite computational resources, ARRGO returns the best solution found:

$$
x_{\mathrm{best}}
=
\arg\max_{x_i\in D_t}f(x_i).
$$

This is a computational result and does not necessarily provide a global
optimality guarantee.

#### Level 2: Asymptotic Convergence

Under the required structural and information conditions,

$$
f_{\mathrm{best}}^{(t)}
\rightarrow
f^*.
$$

This describes the behavior of the algorithm as the refinement process
continues indefinitely.

#### Level 3: Certified Finite-Time Accuracy

Under the additional Certified Mode assumptions,

$$
\Delta_{\mathrm{global}}\leq\epsilon
$$

provides the finite-time guarantee

$$
f_{\mathrm{best}}
\geq
f^*-\epsilon.
$$

These levels must not be conflated.

### What ARRGO Does Not Claim

The theoretical framework does not claim that:

- every continuous function can be solved exactly with a finite number of
  evaluations;
- continuity alone provides a numerical unseen-value bound;
- spatial contraction alone guarantees global optimization;
- a heuristic estimate is automatically a mathematical certificate;
- budget exhaustion implies global optimality;
- objective-value convergence automatically implies optimizer-location
  convergence;
- an estimated Lipschitz constant is automatically a valid Lipschitz bound.

These limitations are part of the mathematical specification of ARRGO.

### Final Theoretical Statement

The theoretical foundation of ARRGO can therefore be summarized as

$$
\boxed{
\begin{aligned}
&\text{Compactness}
+
\text{Continuity}
+
\text{Coverage}
\\
&+
\text{Valid Refinement}
+
\text{Fair Selection}
+
\text{Information Refinement}
\\
&\Longrightarrow
\text{Asymptotic Objective-Value Convergence}.
\end{aligned}
}
$$

For Certified Mode, an additional valid regularity assumption gives

$$
\boxed{
\begin{aligned}
&\text{Valid Lipschitz Bound}
+
\text{Valid Regional Bounds}
\\
&+
\text{Global Coverage}
+
\text{Certified Gap}
\\
&\Longrightarrow
\epsilon\text{-Optimality when }
\Delta_{\mathrm{global}}\leq\epsilon.
\end{aligned}
}
$$

Thus, ARRGO's theoretical framework separates what is guaranteed by
mathematical assumptions from what is produced empirically by finite
computation.

This completes the theoretical foundation required before defining the
concrete ARRGO search and refinement algorithm.